# Part 1: Data Cleaning & Validation

This notebook cleans the intentionally dirty `ohlc_hw` table and writes the result to `ohlc_hw_cleaned`.

**Run from this folder** (`Homeworks/HW2/Shang_Wu/`). `DB_PATH` points to a local working copy at `data/data.db`
(copied from the course database; gitignored, never committed). The raw `ohlc_hw` table is only read, never modified.

In [1]:
import re
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

DB_PATH = Path("data/data.db")
assert DB_PATH.exists(), f"Database not found at {DB_PATH.resolve()} - copy data.db from bCourses"


## 1.1 Load and Inspect the Data

We load the full raw table once and keep it untouched as `raw`; every later step works on copies.

In [2]:
with sqlite3.connect(DB_PATH) as conn:
    raw = pd.read_sql_query("SELECT * FROM ohlc_hw", conn)

print(f"Loaded {len(raw):,} rows x {raw.shape[1]} columns from ohlc_hw")
raw.head()

Loaded 397,711 rows x 8 columns from ohlc_hw


,ts,open,high,low,close,adj_close,volume,ticker
0,1975-04-14,5.4300,5.4800,5.3600,5.4400,1.640000,446392,CAT
1,1978-04-05,4.8400,4.8900,4.8100,4.8800,0.656990,355200,CVX
2,2018-05-25,124.5400,124.7600,121.2200,122.1900,92.220000,8936649,CVX
3,1995-12-07,9.9700,9.9700,9.8300,9.9400,5.880000,3954150,HD
4,1981-07-23,0.1038,0.1038,0.1038,0.1038,0.080013,34451232,AAPL


### Column meanings

| Column | Meaning |
|---|---|
| `ts` | Observation (trading) date, stored as text `YYYY-MM-DD` |
| `ticker` | Stock symbol — **dirty**: some labels are wrapped in HTML markup |
| `open` | Raw (unadjusted) price at the open |
| `high` | Raw intraday high |
| `low` | Raw intraday low |
| `close` | Raw closing price |
| `adj_close` | Vendor close adjusted for splits and dividends (not bounded by raw high/low) |
| `volume` | Shares traded that day |

In [3]:
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   ts         397711 non-null  object 
 1   open       397711 non-null  float64
 2   high       397711 non-null  float64
 3   low        397711 non-null  float64
 4   close      397711 non-null  float64
 5   adj_close  377856 non-null  float64
 6   volume     397711 non-null  int64  
 7   ticker     397711 non-null  object 
dtypes: float64(5), int64(1), object(2)
memory usage: 24.3+ MB


Notes from `.info()`:
- `ts` is an `object` (string) column, so we parse it to datetimes for date work.
- Only `adj_close` has missing cells; the other columns are fully populated (which does not mean their values are valid).

In [4]:
raw.describe()

,open,high,low,close,adj_close,volume
count,397711.000000,397711.000000,397711.000000,397711.000000,377856.000000,3.977110e+05
mean,46.285479,46.747660,45.816279,46.292903,51.652250,3.097773e+07
std,67.132594,67.778210,66.477178,67.141879,194.584045,1.266237e+08
min,0.000000,0.005208,0.004801,0.005208,0.002945,0.000000e+00
25%,4.209938,4.240000,4.160000,4.200000,1.390000,1.730700e+06
50%,19.850000,20.060000,19.630000,19.860000,11.960000,4.648200e+06
75%,57.530000,58.130000,56.950000,57.530000,43.770000,1.102380e+07
max,604.260000,608.630000,597.880000,605.400000,10442.510783,9.276606e+09


### Date range

`ts` is parsed with an explicit format so any malformed date would raise instead of silently becoming `NaT`.

In [5]:
ts = pd.to_datetime(raw["ts"], format="%Y-%m-%d")
print(f"Overall date range: {ts.min().date()} to {ts.max().date()}")
print(f"Distinct dates: {ts.nunique():,}")

Overall date range: 1962-01-02 to 2024-12-06
Distinct dates: 15,868


### Ticker labels

The raw labels include HTML markup (e.g. `<a ...><span style="color: blue">AAPL</span></a>`), so the raw
label count overstates the number of stocks. Here we only *preview* a normalization (strip tags, whitespace, upper-case)
to list the real tickers; the audited normalization happens in the cleaning step.

In [6]:
def normalize_ticker(label: str) -> str:
    """Strip HTML tags and surrounding whitespace, then upper-case a ticker label."""
    return re.sub(r"<[^>]+>", "", label).strip().upper()


print(f"Distinct raw labels: {raw['ticker'].nunique()}")

ticker_preview = raw.assign(ts=ts, ticker_norm=raw["ticker"].map(normalize_ticker))
ticker_summary = (
    ticker_preview.groupby("ticker_norm")
    .agg(
        raw_labels=("ticker", "nunique"),
        rows=("ts", "size"),
        first_date=("ts", "min"),
        last_date=("ts", "max"),
    )
    .sort_index()
)
print(f"Distinct tickers after normalization: {len(ticker_summary)}")
ticker_summary

Distinct raw labels: 150


Distinct tickers after normalization: 30


,raw_labels,rows,first_date,last_date
ticker_norm,,,,
AAPL,5,11942,1980-12-12,2024-10-18
AMGN,5,11290,1983-06-17,2024-10-18
AMZN,5,7458,1997-05-15,2024-12-06
AXP,5,14260,1972-04-03,2024-10-18
BA,5,16980,1962-01-02,2024-10-18
CAT,5,17044,1962-01-02,2024-10-18
CRM,5,5525,2004-06-23,2024-10-18
CSCO,5,9427,1990-02-16,2024-10-18
CVX,5,17073,1962-01-02,2024-12-06


## 1.2 Detect Data Quality Issues

### Missing values in each column

A cell can be "missing" in more than one way, so the detector counts three kinds per column:
- `null`: SQL `NULL`, loaded by pandas as `NaN`/`None`
- `blank`: empty or whitespace-only text (text columns only)
- `non_finite`: `±inf` in a numeric column — not `NaN`, but just as unusable as a price

Missing *cells* are a different check from missing *trading dates* (absent rows), which comes next.

In [7]:
def missing_value_summary(df: pd.DataFrame) -> pd.DataFrame:
    """Count null, blank-text, and non-finite cells in each column of ``df``.

    Returns one row per column with the counts, their total, and the
    total as a percentage of all rows.
    """
    rows = []
    for col in df.columns:
        s = df[col]
        is_null = s.isna()
        if pd.api.types.is_numeric_dtype(s):
            is_blank = pd.Series(False, index=s.index)
            is_non_finite = ~is_null & ~np.isfinite(s.astype(float))
        else:
            is_blank = ~is_null & s.astype(str).str.strip().eq("")
            is_non_finite = pd.Series(False, index=s.index)
        is_missing = is_null | is_blank | is_non_finite
        rows.append(
            {
                "column": col,
                "null": int(is_null.sum()),
                "blank": int(is_blank.sum()),
                "non_finite": int(is_non_finite.sum()),
                "total_missing": int(is_missing.sum()),
                "pct_missing": 100 * is_missing.mean(),
            }
        )
    return pd.DataFrame(rows).set_index("column")


missing_summary = missing_value_summary(raw)
missing_summary.round({"pct_missing": 2})

,null,blank,non_finite,total_missing,pct_missing
column,,,,,
ts,0,0,0,0,0.00
open,0,0,0,0,0.00
high,0,0,0,0,0.00
low,0,0,0,0,0.00
close,0,0,0,0,0.00
adj_close,19855,0,0,19855,4.99
volume,0,0,0,0,0.00
ticker,0,0,0,0,0.00


Missing values are **only in `adj_close`**; every other column is fully populated.
Next we check whether those gaps are concentrated in particular tickers (grouping on the normalized ticker,
since the raw labels split each stock five ways) and over what dates they occur.

In [8]:
adj_missing = ticker_preview.assign(adj_missing=raw["adj_close"].isna())

adj_missing_by_ticker = (
    adj_missing.groupby("ticker_norm")
    .agg(
        rows=("adj_missing", "size"),
        missing_adj_close=("adj_missing", "sum"),
        pct_missing=("adj_missing", "mean"),
    )
    .assign(pct_missing=lambda d: 100 * d["pct_missing"])
    .sort_values("missing_adj_close", ascending=False)
)

missing_dates = adj_missing.loc[adj_missing["adj_missing"], "ts"]
print(
    f"{len(missing_dates):,} rows missing adj_close across "
    f"{(adj_missing_by_ticker['missing_adj_close'] > 0).sum()} of {len(adj_missing_by_ticker)} tickers, "
    f"dated {missing_dates.min().date()} to {missing_dates.max().date()}"
)
adj_missing_by_ticker.round({"pct_missing": 2})

19,855 rows missing adj_close across 30 of 30 tickers, dated 1962-01-04 to 2024-12-06


,rows,missing_adj_close,pct_missing
ticker_norm,,,
MRK,17111,900,5.26
CAT,17044,897,5.26
BA,16980,888,5.23
CVX,17073,886,5.19
KO,17147,859,5.01
JNJ,17097,859,5.02
PG,17108,856,5.00
DIS,17151,856,4.99
HON,17111,850,4.97


### Missing trading days (gaps in the time series)

A missing trading day is an **absent row**, not a missing cell, so it needs a reference calendar to compare against.

**Calendar choice.** Business weekdays (`pd.bdate_range`) would wrongly flag every exchange holiday as a gap. Instead,
each ticker's reference calendar is the set of dates that ticker has in the supplied `ohlc` table, restricted to the
ticker's own history in `ohlc_hw` (its first to last observed date). This means:
- a stock is never expected to trade before it listed or after its data ends, and
- leading/trailing absences are *not* counted as gaps; only interior dates are.

The detector also reports the reverse — dates in `ohlc_hw` that the reference calendar does not have — since an
unexpected extra date is as suspicious as a missing one.

In [9]:
def find_missing_trading_days(observed: pd.DataFrame, reference: pd.DataFrame) -> pd.DataFrame:
    """Return reference dates absent from ``observed``, per ticker.

    Both inputs need ``ticker`` and ``ts`` (datetime) columns; duplicate rows are ignored.
    For each ticker, only reference dates within that ticker's observed first-to-last
    range are expected, so leading/trailing absences are never reported.

    Returns columns ``ticker`` and ``ts``, one row per missing date.
    """
    obs = observed[["ticker", "ts"]].drop_duplicates()
    ref = reference[["ticker", "ts"]].drop_duplicates()

    # Expected window per ticker = its own observed history.
    window = obs.groupby("ticker")["ts"].agg(first="min", last="max").reset_index()
    expected = ref.merge(window, on="ticker", how="inner")
    expected = expected[expected["ts"].between(expected["first"], expected["last"])]

    # Anti-join: expected dates with no matching observed row.
    merged = expected[["ticker", "ts"]].merge(obs, on=["ticker", "ts"], how="left", indicator=True)
    missing = merged.loc[merged["_merge"] == "left_only", ["ticker", "ts"]]
    return missing.sort_values(["ticker", "ts"]).reset_index(drop=True)


def find_unexpected_dates(observed: pd.DataFrame, reference: pd.DataFrame) -> pd.DataFrame:
    """Return observed ``(ticker, ts)`` pairs that are not in the reference calendar."""
    obs = observed[["ticker", "ts"]].drop_duplicates()
    ref = reference[["ticker", "ts"]].drop_duplicates()
    merged = obs.merge(ref, on=["ticker", "ts"], how="left", indicator=True)
    extra = merged.loc[merged["_merge"] == "left_only", ["ticker", "ts"]]
    return extra.sort_values(["ticker", "ts"]).reset_index(drop=True)


def summarize_missing_days(observed: pd.DataFrame, missing: pd.DataFrame) -> pd.DataFrame:
    """One row per observed ticker: its history, number of missing days, and a readable summary.

    Tickers with no gaps are kept with ``missing_days = 0`` so a clean result is reported explicitly.
    """
    summary = observed.groupby("ticker")["ts"].agg(start="min", end="max")
    summary["missing_days"] = missing.groupby("ticker").size().reindex(summary.index, fill_value=0)
    summary["summary"] = [
        f"Ticker {t} is missing {n} trading day{'' if n == 1 else 's'} between {s.date()} and {e.date()}"
        for t, s, e, n in zip(summary.index, summary["start"], summary["end"], summary["missing_days"])
    ]
    return summary

#### Detector check on a tiny example

Before trusting a result of zero, we confirm the detector finds a gap we planted. The reference calendar below
skips 2024-07-04 (a market holiday), and the observed data has one trading day deliberately removed. The detector
should flag that one day and **not** flag the holiday or the dates outside the observed range.

In [10]:
ref_demo = pd.DataFrame({
    "ticker": "DEMO",
    "ts": pd.to_datetime(["2024-07-01", "2024-07-02", "2024-07-03",  # 07-04 holiday: not in calendar
                          "2024-07-05", "2024-07-08", "2024-07-09"]),
})
# Observed data starts a day late (leading absence, not a gap) and is missing 2024-07-05.
obs_demo = ref_demo[~ref_demo["ts"].isin(pd.to_datetime(["2024-07-01", "2024-07-05"]))]

missing_demo = find_missing_trading_days(obs_demo, ref_demo)
assert missing_demo["ts"].tolist() == [pd.Timestamp("2024-07-05")], missing_demo
assert find_unexpected_dates(obs_demo, ref_demo).empty

print(summarize_missing_days(obs_demo, missing_demo)["summary"].iloc[0])
missing_demo

Ticker DEMO is missing 1 trading day between 2024-07-02 and 2024-07-09


,ticker,ts
0,DEMO,2024-07-05


#### Run on `ohlc_hw`

In [11]:
with sqlite3.connect(DB_PATH) as conn:
    ref_calendar = pd.read_sql_query("SELECT DISTINCT ticker, ts FROM ohlc", conn)
ref_calendar["ts"] = pd.to_datetime(ref_calendar["ts"], format="%Y-%m-%d")

# Use the normalized ticker so the five label variants of each stock form one series.
observed_dates = ticker_preview[["ticker_norm", "ts"]].rename(columns={"ticker_norm": "ticker"})

missing_days = find_missing_trading_days(observed_dates, ref_calendar)
unexpected_days = find_unexpected_dates(observed_dates, ref_calendar)
missing_days_summary = summarize_missing_days(observed_dates, missing_days)

print(f"Missing trading days: {len(missing_days)} across {(missing_days_summary['missing_days'] > 0).sum()} tickers")
print(f"Unexpected dates not in reference calendar: {len(unexpected_days)}")
print(f"Tickers in reference but absent from ohlc_hw: "
      f"{sorted(set(ref_calendar['ticker']) - set(observed_dates['ticker'])) or 'none'}")

Missing trading days: 0 across 0 tickers
Unexpected dates not in reference calendar: 0
Tickers in reference but absent from ohlc_hw: none


#### Missing trading days summary

One line per ticker in the form *"Ticker X is missing Y trading days between [start] and [end]"*, where
start/end are that ticker's first and last observed dates (the window the gaps were checked over).
Tickers with zero gaps are listed too, so a clean result is stated rather than implied.

In [12]:
for line in missing_days_summary["summary"]:
    print(line)

missing_days_summary[["start", "end", "missing_days"]].assign(
    start=lambda d: d["start"].dt.date, end=lambda d: d["end"].dt.date
)

Ticker AAPL is missing 0 trading days between 1980-12-12 and 2024-10-18
Ticker AMGN is missing 0 trading days between 1983-06-17 and 2024-10-18
Ticker AMZN is missing 0 trading days between 1997-05-15 and 2024-12-06
Ticker AXP is missing 0 trading days between 1972-04-03 and 2024-10-18
Ticker BA is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker CAT is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker CRM is missing 0 trading days between 2004-06-23 and 2024-10-18
Ticker CSCO is missing 0 trading days between 1990-02-16 and 2024-10-18
Ticker CVX is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticker DIS is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticker GS is missing 0 trading days between 1999-05-04 and 2024-10-18
Ticker HD is missing 0 trading days between 1981-09-22 and 2024-10-18
Ticker HON is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker IBM is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticke

,start,end,missing_days
ticker,,,
AAPL,1980-12-12,2024-10-18,0
AMGN,1983-06-17,2024-10-18,0
AMZN,1997-05-15,2024-12-06,0
AXP,1972-04-03,2024-10-18,0
BA,1962-01-02,2024-10-18,0
CAT,1962-01-02,2024-10-18,0
CRM,2004-06-23,2024-10-18,0
CSCO,1990-02-16,2024-10-18,0
CVX,1962-01-02,2024-12-06,0


**Result: zero missing trading days.** Every ticker in `ohlc_hw` has every date in its reference calendar
across its history, and no dates outside it. The detector demonstrably catches a removed date (above), so
this zero is a real finding, not a silent failure. `ohlc_hw`'s data problems are in the cells (missing `adj_close`,
bad prices, duplicates), not in absent rows.

Note the histories end on different dates (2024-10-18 for most tickers, 2024-12-06 for some). That is a
coverage difference, not a gap; it matters later when choosing a common report window in Part 3.

### Outliers: daily returns

Daily simple returns on **raw close**: `(close - prev_close) / prev_close`.

Returns need exactly one row per `(ticker, ts)` in date order, otherwise `prev_close` could be a duplicate of
the same day, or another ticker's price. So before computing returns we build a working price frame that:
1. normalizes ticker labels (keeping the original label in `ticker_raw` for audit),
2. drops **exact** duplicate records (identical in every field after normalization),
3. asserts that no **conflicting** `(ticker, ts)` keys remain — if any did, we would have to investigate them, not pick one,
4. sorts by ticker and date.

The raw table itself is not changed; the full duplicate audit goes in the data quality report.

In [13]:
DATA_COLS = ["ts", "ticker", "open", "high", "low", "close", "adj_close", "volume"]


def prepare_price_frame(raw_df: pd.DataFrame) -> pd.DataFrame:
    """Return a sorted, de-duplicated working copy with one row per (ticker, ts).

    Normalizes tickers (original kept in ``ticker_raw``), parses ``ts``, drops exact
    duplicates, and raises if conflicting rows share a ``(ticker, ts)`` key.
    """
    df = raw_df.rename(columns={"ticker": "ticker_raw"}).assign(
        ticker=lambda d: d["ticker_raw"].map(normalize_ticker),
        ts=lambda d: pd.to_datetime(d["ts"], format="%Y-%m-%d"),
    )
    # Exact duplicates are judged on the data, not the raw label markup.
    df = df.drop_duplicates(subset=DATA_COLS, keep="first")

    conflicts = df.duplicated(["ticker", "ts"], keep=False)
    if conflicts.any():
        raise ValueError(f"{conflicts.sum()} rows share a (ticker, ts) key with different values - investigate")

    return df.sort_values(["ticker", "ts"]).reset_index(drop=True)


prices = prepare_price_frame(raw)
print(f"Raw rows:                 {len(raw):,}")
print(f"Exact duplicates removed: {len(raw) - len(prices):,}")
print(f"Unique (ticker, ts) rows: {len(prices):,}")

Raw rows:                 397,711
Exact duplicates removed: 29,460
Unique (ticker, ts) rows: 368,251


In [14]:
def calculate_daily_returns(df: pd.DataFrame, price_col: str = "close") -> pd.Series:
    """Simple daily return ``(price - prev_price) / prev_price`` within each ticker.

    ``df`` must be sorted by ticker and date with one row per (ticker, ts).
    The first row of each ticker has no previous price, so its return is NaN.
    Returns are also NaN (undefined) when either price is missing, non-finite,
    or nonpositive, rather than becoming ``inf`` or a meaningless number.
    """
    price = df[price_col]
    prev_price = df.groupby("ticker")[price_col].shift(1)

    valid = np.isfinite(price) & np.isfinite(prev_price) & (price > 0) & (prev_price > 0)
    return ((price - prev_price) / prev_price).where(valid)

Quick check on a tiny hand-made example: rows arrive unsorted and contain two tickers. Each ticker's first
return must be NaN (no leaking of `AAA`'s last price into `BBB`), and a zero price must give NaN, not `inf`.

In [15]:
demo = pd.DataFrame({
    "ticker": ["BBB", "AAA", "AAA", "BBB", "AAA"],
    "ts": pd.to_datetime(["2024-01-02", "2024-01-03", "2024-01-02", "2024-01-03", "2024-01-04"]),
    "close": [50.0, 110.0, 100.0, 0.0, 99.0],
}).sort_values(["ticker", "ts"]).reset_index(drop=True)
demo["return"] = calculate_daily_returns(demo)

expected = [np.nan, 0.10, -0.10, np.nan, np.nan]  # AAA: -, +10%, -10% | BBB: -, zero price -> undefined
assert np.allclose(demo["return"], expected, equal_nan=True), demo
demo

,ticker,ts,close,return
0,AAA,2024-01-02,100.0,NaN
1,AAA,2024-01-03,110.0,0.1
2,AAA,2024-01-04,99.0,-0.1
3,BBB,2024-01-02,50.0,NaN
4,BBB,2024-01-03,0.0,NaN


#### Returns on `ohlc_hw`

In [16]:
prices["return"] = calculate_daily_returns(prices)

n_first = prices["ticker"].nunique()
n_undefined = prices["return"].isna().sum() - n_first
print(f"Returns computed: {prices['return'].notna().sum():,}")
print(f"NaN returns: {prices['return'].isna().sum():,} "
      f"({n_first} first-day rows + {n_undefined} undefined from missing/nonpositive prices)")

return_stats = prices.groupby("ticker")["return"].agg(
    n="count", mean="mean", std="std", min="min", max="max"
)
return_stats

Returns computed: 368,221
NaN returns: 30 (30 first-day rows + 0 undefined from missing/nonpositive prices)


,n,mean,std,min,max
ticker,,,,,
AAPL,11049,0.001074,0.027846,-0.518695,0.332256
AMGN,10420,0.000937,0.023731,-0.198962,0.205120
AMZN,6936,0.001729,0.035298,-0.247661,0.344714
AXP,13251,0.000577,0.022122,-0.261693,0.218823
BA,15783,0.000561,0.021421,-0.238484,0.243186
CAT,15807,0.000521,0.018588,-0.216316,0.147229
CRM,5116,0.001174,0.026448,-0.269327,0.260449
CSCO,8733,0.001064,0.024872,-0.198758,0.243884
CVX,15842,0.000374,0.016185,-0.221248,0.227407


These are **raw-close** returns, so they are a diagnostic, not investment returns: a stock split shows up as a
large negative "return" (e.g. a 2-for-1 split looks like -50%) even though shareholders lost nothing. The
extreme `min`/`max` values above are exactly what the 3-standard-deviation screen in the next step flags for
review — each one must be investigated (split, data error, or real news) before any repair.

#### 3-standard-deviation return screen

Rule, applied **within each ticker**: flag a day when `abs(return - mean_return) > 3 * std_return`, where
`std_return` is the sample standard deviation (`ddof=1`). Taking the absolute value flags both tails
(crashes and spikes).

Edge cases are handled explicitly rather than left to whatever NaN arithmetic produces. Each row gets a
`return_screen` status:

| status | meaning | flag |
|---|---|---|
| `screened` | return and std are both usable | `True` / `False` |
| `return_undefined` | no return (first day, or a missing/nonpositive price) — excluded from mean and std | `<NA>` |
| `std_undefined` | ticker has fewer than 2 defined returns, so `ddof=1` std is undefined | `<NA>` |
| `std_zero` | all of the ticker's returns are identical, so no day can deviate | `<NA>` |

`<NA>` means *not assessable*, which is different from *assessed and fine* (`False`).

This is a **retrospective** screen: the mean and std use the ticker's whole history, including future days, so it is
for reviewing historical data, not a live trading signal. A flag means "look at this day", **not** "delete it".

In [17]:
def flag_return_outliers(df: pd.DataFrame, return_col: str = "return", n_std: float = 3.0) -> pd.DataFrame:
    """Flag returns more than ``n_std`` sample standard deviations from their ticker's mean.

    Undefined (NaN) returns are excluded from each ticker's mean and std. Returns a
    DataFrame aligned to ``df`` with columns ``return_mean``, ``return_std`` (ddof=1),
    ``return_zscore``, ``return_outlier`` (nullable boolean; <NA> when not assessable),
    and ``return_screen`` (status explaining the flag).
    """
    r = df[return_col].where(np.isfinite(df[return_col]))  # treat +/-inf as undefined too
    grouped = r.groupby(df["ticker"])
    mean = grouped.transform("mean")         # pandas skips NaN
    std = grouped.transform("std", ddof=1)   # NaN when a ticker has < 2 defined returns

    status = pd.Series("screened", index=df.index)
    status[std == 0] = "std_zero"
    status[std.isna()] = "std_undefined"
    status[r.isna()] = "return_undefined"  # row-level reason takes priority

    screened = status == "screened"
    zscore = ((r - mean) / std).where(screened)
    outlier = pd.Series(pd.NA, index=df.index, dtype="boolean")
    outlier[screened] = (r - mean).abs()[screened] > n_std * std[screened]

    return pd.DataFrame({
        "return_mean": mean,
        "return_std": std,
        "return_zscore": zscore,
        "return_outlier": outlier,
        "return_screen": status,
    })

Check on hand-made cases, since the real data may not contain every edge case:
- `AAA`: 20 small returns plus one +50% and one -50% day — both tails must be flagged, nothing else.
  (A lone outlier needs enough other points to exceed 3 std: with *n* returns the largest possible z-score is (*n*-1)/√*n*.)
- `BBB`: constant returns — std is exactly 0.
- `CCC`: a single defined return — sample std is undefined.
- A NaN return in `AAA` must be excluded, not counted as zero.

In [18]:
demo = pd.DataFrame({
    "ticker": ["AAA"] * 23 + ["BBB"] * 5 + ["CCC"] * 2,
    "return": [0.01, -0.01] * 10 + [0.5, -0.5, np.nan]
              + [0.002] * 5
              + [np.nan, 0.03],
})
demo_flags = flag_return_outliers(demo)
demo = demo.join(demo_flags[["return_zscore", "return_outlier", "return_screen"]])

aaa = demo[demo["ticker"] == "AAA"]
assert aaa.loc[aaa["return_outlier"].fillna(False), "return"].tolist() == [0.5, -0.5]
assert aaa["return_screen"].iloc[-1] == "return_undefined" and pd.isna(aaa["return_outlier"].iloc[-1])
assert (demo.loc[demo["ticker"] == "BBB", "return_screen"] == "std_zero").all()
assert demo.loc[demo["ticker"] == "CCC", "return_screen"].tolist() == ["return_undefined", "std_undefined"]

demo.iloc[18:]

,ticker,return,return_zscore,return_outlier,return_screen
18,AAA,0.010,0.064678,False,screened
19,AAA,-0.010,-0.064678,False,screened
20,AAA,0.500,3.233909,True,screened
21,AAA,-0.500,-3.233909,True,screened
22,AAA,NaN,NaN,<NA>,return_undefined
23,BBB,0.002,NaN,<NA>,std_zero
24,BBB,0.002,NaN,<NA>,std_zero
25,BBB,0.002,NaN,<NA>,std_zero
26,BBB,0.002,NaN,<NA>,std_zero
27,BBB,0.002,NaN,<NA>,std_zero


#### Screen `ohlc_hw` returns

In [19]:
prices = prices.drop(columns=["return_mean", "return_std", "return_zscore", "return_outlier", "return_screen"],
                     errors="ignore").join(flag_return_outliers(prices))

print(prices["return_screen"].value_counts().to_string())
print()
flagged = prices[prices["return_outlier"].fillna(False)]
print(f"Flagged: {len(flagged):,} of {prices['return_outlier'].notna().sum():,} screened returns "
      f"({len(flagged) / prices['return_outlier'].notna().sum():.2%}); "
      f"{(flagged['return'] > flagged['return_mean']).sum():,} upper tail, "
      f"{(flagged['return'] < flagged['return_mean']).sum():,} lower tail")

outlier_counts = (
    prices.groupby("ticker")
    .agg(
        screened=("return_outlier", "count"),
        flagged=("return_outlier", "sum"),
        std=("return_std", "first"),
    )
    .assign(
        upper=flagged.loc[flagged["return"] > flagged["return_mean"]].groupby("ticker").size(),
        lower=flagged.loc[flagged["return"] < flagged["return_mean"]].groupby("ticker").size(),
        threshold=lambda d: 3 * d["std"],
    )
    .fillna({"upper": 0, "lower": 0})
    .astype({"upper": int, "lower": int, "flagged": int})
)
outlier_counts

return_screen
screened            368221
return_undefined        30

Flagged: 5,056 of 368,221 screened returns (1.37%); 2,830 upper tail, 2,226 lower tail


,screened,flagged,std,upper,lower,threshold
ticker,,,,,,
AAPL,11049,143,0.027846,88,55,0.083538
AMGN,10420,172,0.023731,96,76,0.071193
AMZN,6936,147,0.035298,91,56,0.105895
AXP,13251,199,0.022122,106,93,0.066366
BA,15783,210,0.021421,119,91,0.064262
CAT,15807,221,0.018588,116,105,0.055763
CRM,5116,82,0.026448,50,32,0.079343
CSCO,8733,148,0.024872,75,73,0.074617
CVX,15842,178,0.016185,101,77,0.048556


The most extreme flagged days, by absolute z-score — the first candidates to investigate (split vs. data error vs.
real news) in the cleaning step:

In [20]:
review_cols = ["ticker", "ts", "open", "high", "low", "close", "adj_close", "return", "return_zscore"]
flagged.reindex(flagged["return_zscore"].abs().sort_values(ascending=False).index)[review_cols].head(15)

,ticker,ts,open,high,low,close,adj_close,return,return_zscore
318332,TRV,1976-10-28,9.22000,9.22000,9.22000,9.22000,NaN,1.000000,41.244530
319538,TRV,1981-08-10,12.06000,12.06000,12.06000,12.06000,4.200000,1.000000,41.244530
318873,TRV,1978-12-20,8.03000,8.03000,8.03000,8.03000,2.800000,0.997512,41.141873
213788,MCD,1968-01-02,0.17490,0.17490,0.17490,0.17490,0.075921,-0.753620,-40.720008
296654,PG,2000-03-07,28.78000,30.89000,28.41000,29.78000,15.580000,-0.313667,-23.196018
319539,TRV,1981-08-11,6.03000,6.03000,6.03000,6.03000,1.570000,-0.500000,-20.657761
293525,PG,1987-10-19,5.15000,5.15000,3.72000,3.81000,1.520000,-0.277040,-20.490831
318333,TRV,1976-10-29,4.69000,4.69000,4.69000,4.69000,0.997070,-0.491323,-20.299686
318874,TRV,1978-12-21,4.09000,4.09000,4.09000,4.09000,0.924090,-0.490660,-20.272317
4999,AAPL,2000-09-29,0.50335,0.51786,0.45313,0.45982,0.388310,-0.518695,-18.665793


**Reading the result.** Daily returns are fat-tailed, so a 3-std rule flags far more than the ~0.27% of days a normal
distribution would predict — many flags are likely genuine market moves (e.g. the 1987-10-19 crash days in the table above), not errors.
That is why this is a review flag: rows stay in the data with `return_outlier` recorded, and only days later
**confirmed** as data errors get repaired.

#### Impossible prices: close outside the day's range

The close is one of the day's trades, so it must lie within `[low, high]`. A day with `high < close` or
`low > close` is impossible, whatever happened in the market — it can only be a data error.

The two conditions get **separate** flags so the report can say which bound was broken (a day could break both).
Comparisons are exact: prices are stored to a few decimals, so no rounding tolerance is needed, and a looser
test would hide real errors. If any of `high`, `low`, `close` is missing the check is *not assessable*
(`<NA>`), rather than silently passing because `NaN < x` is `False`.

In [21]:
def flag_close_outside_range(df: pd.DataFrame) -> pd.DataFrame:
    """Flag days where the close lies outside the day's [low, high] range.

    Returns a DataFrame aligned to ``df`` with nullable boolean columns
    ``high_below_close`` (high < close) and ``low_above_close`` (low > close);
    each is <NA> when any of high, low, close is missing.
    """
    assessable = df[["high", "low", "close"]].notna().all(axis=1)
    flags = pd.DataFrame({
        "high_below_close": df["high"] < df["close"],
        "low_above_close": df["low"] > df["close"],
    }).astype("boolean")
    flags[~assessable] = pd.NA
    return flags

Check on injected examples, since (as shown below) the database contains none: one valid day, one with
`high < close`, one with `low > close`, a boundary day where close equals the high (valid), and a day with a
missing low.

In [22]:
demo = pd.DataFrame({
    "case": ["valid", "high < close", "low > close", "close == high (valid)", "missing low"],
    "high": [10.0, 10.0, 10.0, 10.0, 10.0],
    "low": [9.0, 9.0, 9.8, 9.0, np.nan],
    "close": [9.5, 10.4, 9.5, 10.0, 9.5],
})
demo = demo.join(flag_close_outside_range(demo))

assert demo["high_below_close"].tolist()[:4] == [False, True, False, False]
assert demo["low_above_close"].tolist()[:4] == [False, False, True, False]
assert demo.iloc[4][["high_below_close", "low_above_close"]].isna().all()
demo

,case,high,low,close,high_below_close,low_above_close
0,valid,10.0,9.0,9.5,False,False
1,high < close,10.0,9.0,10.4,True,False
2,low > close,10.0,9.8,9.5,False,True
3,close == high (valid),10.0,9.0,10.0,False,False
4,missing low,10.0,NaN,9.5,<NA>,<NA>


#### Check `ohlc_hw`

In [23]:
prices = prices.drop(columns=["high_below_close", "low_above_close"], errors="ignore").join(
    flag_close_outside_range(prices)
)

for col in ["high_below_close", "low_above_close"]:
    print(f"{col:17}: {int(prices[col].sum())} flagged, {prices[col].isna().sum()} not assessable")

close_range_issues = prices[prices["high_below_close"] | prices["low_above_close"]]
print(f"Days with close outside [low, high]: {len(close_range_issues)}")
close_range_issues[["ticker", "ts", "open", "high", "low", "close", "high_below_close", "low_above_close"]]

high_below_close : 0 flagged, 0 not assessable
low_above_close  : 0 flagged, 0 not assessable
Days with close outside [low, high]: 0


,ticker,ts,open,high,low,close,high_below_close,low_above_close


**Result: zero days** have the close outside the day's range, and every row was assessable. The detector
catches both violations in the injected example, so this zero is a genuine finding. The other price-bound
checks (open within the range, `low <= high`, nonpositive prices — the data has a few zero opens) come next.

#### Impossible prices: nonpositive or non-finite

An equity price must be a finite number greater than zero. This applies to all five price columns,
**including `adj_close`**. Each bad cell gets one problem label:

| problem | test |
|---|---|
| `missing` | `NaN` (already counted in the missing-values check; repeated here so price validity is complete) |
| `non_finite` | `+inf` / `-inf` |
| `nonpositive` | `<= 0` |

The output is a long table, one row per bad *cell* (`ticker, ts, column, value, problem`), because one day can have
several bad prices and the data quality report needs each one.

**What this check does not do:** apply raw OHLC bounds to `adj_close`. Adjusted close is the raw close rescaled for
later splits and dividends, so it is routinely *below* the day's raw low (or above the high). Only its positivity and
finiteness are checked here; its consistency with raw close is a separate question for the cleaning step.

In [24]:
PRICE_COLS = ["open", "high", "low", "close", "adj_close"]


def find_invalid_prices(df: pd.DataFrame, price_cols: list[str] = PRICE_COLS) -> pd.DataFrame:
    """List every missing, non-finite, or nonpositive price cell.

    Returns one row per bad cell with columns ``ticker``, ``ts``, ``column``,
    ``value``, and ``problem`` (``missing`` / ``non_finite`` / ``nonpositive``).
    """
    long = df.melt(id_vars=["ticker", "ts"], value_vars=price_cols, var_name="column", value_name="value")
    v = long["value"]
    problem = np.select(
        [v.isna(), ~np.isfinite(v), v <= 0],  # order matters: NaN first, then inf, then sign
        ["missing", "non_finite", "nonpositive"],
        default="",
    )
    return long.assign(problem=problem).loc[lambda d: d["problem"] != ""].reset_index(drop=True)

Check on injected examples, covering cases the database may not have (infinities, negative and
nonpositive adjusted close):

In [25]:
demo = pd.DataFrame({
    "ticker": ["AAA", "AAA", "AAA", "AAA"],
    "ts": pd.to_datetime(["2024-01-02", "2024-01-03", "2024-01-04", "2024-01-05"]),
    "open":      [10.0,   0.0,  10.0,    10.0],
    "high":      [11.0,  11.0, np.inf,   11.0],
    "low":       [ 9.0,   9.0,   9.0,    -1.0],
    "close":     [10.5,  10.5,  10.5,    10.5],
    "adj_close": [ 5.0,  np.nan, 0.0,  -np.inf],  # 5.0 is below the raw low: valid for adjusted close
})
demo_invalid = find_invalid_prices(demo)

assert set(map(tuple, demo_invalid[["ts", "column", "problem"]].astype(str).values)) == {
    ("2024-01-03", "open", "nonpositive"),
    ("2024-01-03", "adj_close", "missing"),
    ("2024-01-04", "high", "non_finite"),
    ("2024-01-04", "adj_close", "nonpositive"),
    ("2024-01-05", "low", "nonpositive"),
    ("2024-01-05", "adj_close", "non_finite"),
}
demo_invalid

,ticker,ts,column,value,problem
0,AAA,2024-01-03,open,0.0,nonpositive
1,AAA,2024-01-04,high,inf,non_finite
2,AAA,2024-01-05,low,-1.0,nonpositive
3,AAA,2024-01-03,adj_close,NaN,missing
4,AAA,2024-01-04,adj_close,0.0,nonpositive
5,AAA,2024-01-05,adj_close,-inf,non_finite


#### Check `ohlc_hw`

In [26]:
invalid_prices = find_invalid_prices(prices)

print("Bad price cells by column and problem:")
print(
    invalid_prices.pivot_table(index="column", columns="problem", values="ts", aggfunc="count", fill_value=0)
    .reindex(PRICE_COLS, fill_value=0)
    .to_string()
)
print(f"\nRows with at least one bad price: {invalid_prices[['ticker', 'ts']].drop_duplicates().shape[0]:,}")

# Everything except the already-known missing adj_close values:
invalid_prices[invalid_prices["problem"] != "missing"]

Bad price cells by column and problem:
problem    missing  nonpositive
column                         
open             0            5
high             0            0
low              0            0
close            0            0
adj_close    18374            0

Rows with at least one bad price: 18,379


,ticker,ts,column,value,problem
0,CVX,1962-01-02,open,0.0,nonpositive
1,JNJ,1962-01-02,open,0.0,nonpositive
2,MCD,1966-07-05,open,0.0,nonpositive
3,MRK,1962-01-02,open,0.0,nonpositive
4,PG,1962-01-02,open,0.0,nonpositive


Are the zero opens a vendor artifact or an injected error? Compare them with the supplied `ohlc` reference table,
and note where they fall in each ticker's history:

In [27]:
zero_opens = invalid_prices[(invalid_prices["column"] == "open") & (invalid_prices["problem"] == "nonpositive")]

with sqlite3.connect(DB_PATH) as conn:
    ref_open = pd.read_sql_query("SELECT ticker, ts, open AS ref_open FROM ohlc", conn)
ref_open["ts"] = pd.to_datetime(ref_open["ts"], format="%Y-%m-%d")

first_dates = prices.groupby("ticker")["ts"].min().rename("first_date")
zero_opens.merge(ref_open, on=["ticker", "ts"], how="left").join(first_dates, on="ticker").assign(
    is_first_day=lambda d: d["ts"] == d["first_date"]
)

,ticker,ts,column,value,problem,ref_open,first_date,is_first_day
0,CVX,1962-01-02,open,0.0,nonpositive,0.0,1962-01-02,True
1,JNJ,1962-01-02,open,0.0,nonpositive,0.0,1962-01-02,True
2,MCD,1966-07-05,open,0.0,nonpositive,0.0,1966-07-05,True
3,MRK,1962-01-02,open,0.0,nonpositive,0.0,1962-01-02,True
4,PG,1962-01-02,open,0.0,nonpositive,0.0,1962-01-02,True


Why raw bounds must not be applied to `adj_close` — testing it against the raw range would flag most of the dataset:

In [28]:
has_adj = prices["adj_close"].notna()
outside = has_adj & ((prices["adj_close"] < prices["low"]) | (prices["adj_close"] > prices["high"]))
print(f"adj_close outside raw [low, high]: {outside.sum():,} of {has_adj.sum():,} rows ({outside.sum() / has_adj.sum():.1%})")

adj_close outside raw [low, high]: 332,095 of 349,877 rows (94.9%)


**Result.**
- **Missing:** only `adj_close` (the gaps found earlier).
- **Non-finite:** none, in any column.
- **Nonpositive:** 5 zero `open` values, and nothing else — no zero or negative `high`, `low`, `close`, or `adj_close`.

All five zero opens are on the ticker's **first day of data**, and the reference `ohlc` table has the same zeros, so they
are a vendor placeholder for "open not recorded" rather than errors injected into `ohlc_hw`. They are still invalid
prices and must be repaired or quarantined in the cleaning step. Forward-fill cannot help on a first day (there is no
earlier price), so the options are to derive the open from that day's other prices or to quarantine the row.

Most `adj_close` values sit outside the raw range, which is expected and is why it is checked only for positivity
and finiteness.

### Inconsistencies

#### OHLC bounds: `low <= open <= high` and `low <= close <= high`

The open and close are trades during the day, so both must lie within `[low, high]`. Each of the four ways to
break this gets its own flag, so the report can say exactly which bound failed. `low > high` is checked too:
if the range itself is inverted, every other bound test is meaningless for that day.

As before, comparisons are exact and a row with any missing OHLC value is *not assessable* (`<NA>`).
The two close flags are the same tests as the earlier "impossible close" check (`close > high` is `high < close`),
and the notebook asserts that they agree.

In [29]:
def flag_ohlc_bounds(df: pd.DataFrame) -> pd.DataFrame:
    """Flag violations of low <= open <= high, low <= close <= high, and low <= high.

    Returns nullable boolean columns ``open_below_low``, ``open_above_high``,
    ``close_below_low``, ``close_above_high``, ``low_above_high``, and
    ``ohlc_inconsistent`` (any of them); all are <NA> when any of open, high,
    low, close is missing.
    """
    o, h, l, c = df["open"], df["high"], df["low"], df["close"]
    flags = pd.DataFrame({
        "open_below_low": o < l,
        "open_above_high": o > h,
        "close_below_low": c < l,
        "close_above_high": c > h,
        "low_above_high": l > h,
    })
    flags["ohlc_inconsistent"] = flags.any(axis=1)
    flags = flags.astype("boolean")
    flags[~df[["open", "high", "low", "close"]].notna().all(axis=1)] = pd.NA
    return flags

Check on injected examples — one valid bar, one per violation, a boundary bar (open == low), and a missing value:

In [30]:
demo = pd.DataFrame({
    "case": ["valid", "open < low", "open > high", "close < low", "close > high", "low > high",
             "open == low (valid)", "missing open"],
    "open":  [10.0,  8.5, 11.5, 10.0, 10.0, 10.0,  9.0, np.nan],
    "high":  [11.0, 11.0, 11.0, 11.0, 11.0,  9.0, 11.0, 11.0],
    "low":   [ 9.0,  9.0,  9.0,  9.0,  9.0, 11.0,  9.0,  9.0],
    "close": [10.5, 10.5, 10.5,  8.0, 12.0, 10.0, 10.5, 10.5],
})
demo_flags = flag_ohlc_bounds(demo)

flag_cols = ["open_below_low", "open_above_high", "close_below_low", "close_above_high", "low_above_high"]
# Row i (1..5) should trip exactly flag i-1; valid and boundary rows trip nothing.
expected = np.zeros((7, 5), dtype=bool)
for i in range(5):
    expected[i + 1, i] = True
expected[5, :] = True  # with low > high, open and close are below the low AND above the high
assert (demo_flags[flag_cols].iloc[:7].to_numpy(dtype=bool) == expected).all(), demo_flags
assert demo_flags.iloc[7].isna().all()
demo.join(demo_flags)

,case,open,high,low,close,open_below_low,open_above_high,close_below_low,close_above_high,low_above_high,ohlc_inconsistent
0,valid,10.0,11.0,9.0,10.5,False,False,False,False,False,False
1,open < low,8.5,11.0,9.0,10.5,True,False,False,False,False,True
2,open > high,11.5,11.0,9.0,10.5,False,True,False,False,False,True
3,close < low,10.0,11.0,9.0,8.0,False,False,True,False,False,True
4,close > high,10.0,11.0,9.0,12.0,False,False,False,True,False,True
5,low > high,10.0,9.0,11.0,10.0,True,True,True,True,True,True
6,open == low (valid),9.0,11.0,9.0,10.5,False,False,False,False,False,False
7,missing open,NaN,11.0,9.0,10.5,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


#### Check `ohlc_hw`

In [31]:
prices = prices.drop(columns=flag_cols + ["ohlc_inconsistent"], errors="ignore").join(flag_ohlc_bounds(prices))

# Consistency with the earlier close-range check.
assert prices["close_above_high"].equals(prices["high_below_close"])
assert prices["close_below_low"].equals(prices["low_above_close"])

print(prices[flag_cols + ["ohlc_inconsistent"]].sum().astype(int).to_string())
print(f"not assessable: {prices['ohlc_inconsistent'].isna().sum()}")

ohlc_bound_issues = prices[prices["ohlc_inconsistent"]]
ohlc_bound_issues[["ticker", "ts", "open", "high", "low", "close"] + flag_cols]

open_below_low       5
open_above_high      0
close_below_low      0
close_above_high     0
low_above_high       0
ohlc_inconsistent    5
not assessable: 0


,ticker,ts,open,high,low,close,open_below_low,open_above_high,close_below_low,close_above_high,low_above_high
87103,CVX,1962-01-02,0.0,3.30000,3.24000,3.30000,True,False,False,False,False
167733,JNJ,1962-01-02,0.0,0.22338,0.22222,0.22338,True,False,False,False,False
213412,MCD,1966-07-05,0.0,0.27366,0.26749,0.26955,True,False,False,False,False
243893,MRK,1962-01-02,0.0,0.39703,0.39317,0.39537,True,False,False,False,False
287040,PG,1962-01-02,0.0,1.44000,1.42000,1.43000,True,False,False,False,False


**Result.** The only violations are the **5 zero opens** found in the previous check — a zero open is necessarily
below the low. No day has the open above the high, the close outside the range, or an inverted range. So
these five rows are the full list of OHLC inconsistencies, and they share one root cause (an unrecorded first-day open).

#### Zero-volume days

A day with zero volume means no shares traded, so no new price was set. A data feed often fills such days by
**repeating the last price**: open = high = low = close, often equal to the previous close. Those prices are *stale*
— they carry no new information, and they produce artificial zero returns.

Each zero-volume day is classified by two pieces of supporting evidence:
- `flat_bar`: open = high = low = close
- `close_unchanged`: close equals the ticker's previous close

Zero volume is a **review flag**, not an error by itself: thinly traded stocks (especially decades ago) did
have genuine no-trade days. Negative volume, by contrast, is impossible and is checked too.

In [32]:
def flag_zero_volume(df: pd.DataFrame) -> pd.DataFrame:
    """Flag zero- and negative-volume days, with stale-price evidence for zero-volume days.

    ``df`` must be sorted by ticker and date. Returns ``zero_volume``, ``negative_volume``,
    ``flat_bar`` (open == high == low == close), and ``close_unchanged`` (close equals
    the ticker's previous close; False on a ticker's first day).
    """
    prev_close = df.groupby("ticker")["close"].shift(1)
    return pd.DataFrame({
        "zero_volume": df["volume"] == 0,
        "negative_volume": df["volume"] < 0,
        "flat_bar": (df["open"] == df["high"]) & (df["high"] == df["low"]) & (df["low"] == df["close"]),
        "close_unchanged": df["close"] == prev_close,
    })

In [33]:
demo = pd.DataFrame({
    "ticker": ["AAA"] * 4 + ["BBB"],
    "open":   [10.0, 10.0, 10.0, 10.2, 5.0],
    "high":   [10.5, 10.0, 10.3, 10.4, 5.0],
    "low":    [ 9.8, 10.0,  9.9, 10.1, 5.0],
    "close":  [10.0, 10.0, 10.1, 10.3, 5.0],
    "volume": [1000,    0,    0,   -5,    0],
})
demo = demo.join(flag_zero_volume(demo))

assert demo["zero_volume"].tolist() == [False, True, True, False, True]
assert demo["negative_volume"].tolist() == [False, False, False, True, False]
assert demo["flat_bar"].tolist() == [False, True, False, False, True]
assert demo["close_unchanged"].tolist() == [False, True, False, False, False]  # BBB's first day has no previous close
demo

,ticker,open,high,low,close,volume,zero_volume,negative_volume,flat_bar,close_unchanged
0,AAA,10.0,10.5,9.8,10.0,1000,False,False,False,False
1,AAA,10.0,10.0,10.0,10.0,0,True,False,True,True
2,AAA,10.0,10.3,9.9,10.1,0,True,False,False,False
3,AAA,10.2,10.4,10.1,10.3,-5,False,True,False,False
4,BBB,5.0,5.0,5.0,5.0,0,True,False,True,False


In [34]:
prices = prices.drop(columns=["zero_volume", "negative_volume", "flat_bar", "close_unchanged"], errors="ignore").join(
    flag_zero_volume(prices)
)
zero_vol = prices[prices["zero_volume"]]

print(f"Zero-volume days:     {len(zero_vol)}")
print(f"Negative-volume days: {int(prices['negative_volume'].sum())}")
print(f"  ...of zero-volume days, flat bar:        {int(zero_vol['flat_bar'].sum())}")
print(f"  ...of zero-volume days, close unchanged: {int(zero_vol['close_unchanged'].sum())}")
print(f"Date range of zero-volume days: {zero_vol['ts'].min().date()} to {zero_vol['ts'].max().date()}")

with sqlite3.connect(DB_PATH) as conn:
    ref_zero = pd.read_sql_query("SELECT ticker, ts FROM ohlc WHERE volume = 0", conn)
ref_zero["ts"] = pd.to_datetime(ref_zero["ts"], format="%Y-%m-%d")
same_as_ref = zero_vol[["ticker", "ts"]].merge(ref_zero, how="inner")
print(f"Also zero volume in reference ohlc: {len(same_as_ref)} of {len(zero_vol)} (reference has {len(ref_zero)})")

zero_vol.groupby("ticker").agg(
    days=("ts", "size"),
    first=("ts", "min"),
    last=("ts", "max"),
    flat_bar=("flat_bar", "sum"),
    close_unchanged=("close_unchanged", "sum"),
).sort_values("days", ascending=False)

Zero-volume days:     75
Negative-volume days: 0
  ...of zero-volume days, flat bar:        72
  ...of zero-volume days, close unchanged: 65
Date range of zero-volume days: 1962-01-02 to 1982-09-24
Also zero volume in reference ohlc: 75 of 75 (reference has 75)


,days,first,last,flat_bar,close_unchanged
ticker,,,,,
WMT,52,1972-10-10,1979-05-24,52,51
JNJ,11,1962-01-02,1967-06-14,8,9
IBM,3,1973-10-09,1982-01-08,3,0
TRV,3,1976-10-28,1981-08-10,3,0
HON,2,1982-09-23,1982-09-24,2,2
AXP,1,1972-12-14,1972-12-14,1,0
MRK,1,1963-06-25,1963-06-25,1,1
NKE,1,1981-08-10,1981-08-10,1,1
SHW,1,1982-02-12,1982-02-12,1,1


Zero-volume days that are *not* flat bars — the less typical cases, worth a direct look:

In [35]:
zero_vol.loc[~zero_vol["flat_bar"], ["ticker", "ts", "open", "high", "low", "close", "adj_close", "volume", "return"]]

,ticker,ts,open,high,low,close,adj_close,volume,return
167733,JNJ,1962-01-02,0.00000,0.22338,0.22222,0.22338,0.064012,0,NaN
167807,JNJ,1962-04-17,0.21875,0.21875,0.21759,0.21875,0.062834,0,0.00000
167921,JNJ,1962-09-28,0.15741,0.15914,0.15741,0.15914,0.046002,0,0.01099


**Interpretation.** All zero-volume days fall in the early decades of the data (none after the early 1980s), almost all
are flat bars, and they match the reference `ohlc` table exactly, so they are vendor-supplied history rather than
injected errors. The pattern is consistent with stale, carried-forward quotes on no-trade days. No volume is negative.

**Cross-check with the return screen:** TRV's zero-volume flat bars on 1976-10-28 and 1981-08-10 are two of the
+100% return outliers, each followed by a ~-50% day. A doubled price on a day with no trading is strong evidence of
a bad quote rather than a real move, so those days are prime candidates for repair in the cleaning step.

These rows are kept and flagged (severity: low), not deleted: removing them would create gaps in the trading
calendar, and their prices, while stale, are not impossible. The zero returns they produce are noted as a minor bias
in early-period volatility.

### Data Quality Report

All the detectors above are combined into one table: `ticker | issue_type | issue_date | description | severity`.

- **One row per issue, not per observation.** A day with several problems gets several rows. For example, a zero open is
  both an `invalid_price` and an `ohlc_bounds` violation.
- **Missing trading days are their own rows**, dated with the absent date (there are none in this data, but the rows
  would appear if the detector found any).
- This is the **diagnosis before any repair**. It is saved as `quality_report_before` and is not overwritten; after
  cleaning, the validation reruns the same detectors on the cleaned data.

#### Severity

| severity | meaning | issue types |
|---|---|---|
| **high** | The value is impossible, or a required price is unusable, so the row cannot be used as-is | `invalid_price` (nonpositive/non-finite), `ohlc_bounds`, `missing_trading_day`, `conflicting_record` |
| **medium** | The row is usable for some purposes but will bias results until fixed, or it is an extreme move that needs investigating | `missing_value` (missing `adj_close` breaks adjusted-return calculations), `duplicate_record` (would double-count days and add fake zero returns), `return_outlier` with \|z\| ≥ 10 |
| **low** | Plausible data that is worth a look but probably correct | `return_outlier` with 3 < \|z\| < 10, `zero_volume` |

Why split return outliers at \|z\| = 10: daily returns are fat-tailed, so moves of 3–10 std are seen in real crashes and
earnings surprises. Moves of 10 std or more are rare enough that each one should be explained (split, data error, or
documented news) before it is trusted.

In [36]:
REPORT_COLS = ["ticker", "issue_type", "issue_date", "description", "severity"]


def _issues(df: pd.DataFrame, issue_type: str, description, severity) -> pd.DataFrame:
    """Build report rows from ``df`` (needs ``ticker`` and ``ts``).

    ``description`` and ``severity`` may be constants or Series aligned to ``df``.
    """
    return pd.DataFrame({
        "ticker": df["ticker"].to_numpy(),
        "issue_type": issue_type,
        "issue_date": df["ts"].to_numpy(),
        "description": description if isinstance(description, str) else np.asarray(description),
        "severity": severity if isinstance(severity, str) else np.asarray(severity),
    })


def duplicate_audit(raw_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Find repeated records in the raw table after ticker normalization.

    Returns ``(exact, conflicting)``: exact duplicates counted per (ticker, ts) with
    the number of extra copies, and (ticker, ts) keys whose rows differ in any value.
    """
    df = raw_df.assign(
        ticker=raw_df["ticker"].map(normalize_ticker),
        ts=pd.to_datetime(raw_df["ts"], format="%Y-%m-%d"),
    )
    extra = df[df.duplicated(subset=DATA_COLS, keep="first")]
    exact = extra.groupby(["ticker", "ts"]).size().rename("extra_copies").reset_index()

    distinct = df.drop_duplicates(subset=DATA_COLS)
    keys = distinct.groupby(["ticker", "ts"]).size()
    conflicting = keys[keys > 1].rename("versions").reset_index()
    return exact, conflicting


def build_quality_report(raw_df: pd.DataFrame, prices_df: pd.DataFrame, missing_days_df: pd.DataFrame) -> pd.DataFrame:
    """Combine every detector into one Data Quality Report (one row per issue)."""
    parts = []

    # Missing and invalid price cells.
    bad = find_invalid_prices(prices_df)
    missing = bad[bad["problem"] == "missing"]
    parts.append(_issues(missing, "missing_value", "missing " + missing["column"], "medium"))
    invalid = bad[bad["problem"] != "missing"]
    parts.append(_issues(
        invalid, "invalid_price",
        invalid["problem"] + " " + invalid["column"] + " (" + invalid["value"].astype(str) + ")",
        "high",
    ))

    # Absent rows.
    parts.append(_issues(missing_days_df, "missing_trading_day", "date in reference calendar but no row", "high"))

    # Duplicates.
    exact, conflicting = duplicate_audit(raw_df)
    parts.append(_issues(exact, "duplicate_record",
                         exact["extra_copies"].map(lambda n: f"{n} exact duplicate {'copy' if n == 1 else 'copies'} removed"), "medium"))
    parts.append(_issues(conflicting, "conflicting_record",
                         conflicting["versions"].astype(str) + " differing rows share this (ticker, ts)", "high"))

    # OHLC bounds: one row per violated bound.
    bound_text = {
        "open_below_low": "open < low",
        "open_above_high": "open > high",
        "close_below_low": "close < low",
        "close_above_high": "close > high",
        "low_above_high": "low > high",
    }
    for col, text in bound_text.items():
        hit = prices_df[prices_df[col].fillna(False)]
        detail = hit.apply(lambda r: f"{text} (O={r.open}, H={r.high}, L={r.low}, C={r.close})", axis=1)
        parts.append(_issues(hit, "ohlc_bounds", detail, "high"))

    # Return outliers (review flags).
    out = prices_df[prices_df["return_outlier"].fillna(False)]
    parts.append(_issues(
        out, "return_outlier",
        out.apply(lambda r: f"return {r['return']:+.2%} is {r.return_zscore:+.1f} std from ticker mean", axis=1),
        np.where(out["return_zscore"].abs() >= 10, "medium", "low"),
    ))

    # Zero volume, with stale-price evidence.
    zv = prices_df[prices_df["zero_volume"]]
    evidence = np.select(
        [zv["flat_bar"] & zv["close_unchanged"], zv["flat_bar"], zv["close_unchanged"]],
        ["; flat bar, close unchanged (likely stale)", "; flat bar", "; close unchanged"],
        default="",
    )
    parts.append(_issues(zv, "zero_volume", "volume = 0" + pd.Series(evidence, index=zv.index), "low"))

    report = pd.concat(parts, ignore_index=True)[REPORT_COLS]
    report["severity"] = pd.Categorical(report["severity"], categories=["high", "medium", "low"], ordered=True)
    return report.sort_values(["severity", "ticker", "issue_date", "issue_type"]).reset_index(drop=True)

Check on a tiny injected dataset that every issue type reaches the report, a single observation can carry
several issues, and a removed date shows up as its own row:

In [37]:
demo_raw = pd.DataFrame({
    "ts": ["2024-01-02", "2024-01-03", "2024-01-03", "2024-01-05", "2024-01-08"],
    "ticker": ["<b>aaa</b>", " AAA", "AAA", "AAA", "AAA"],
    "open": [10.0, 0.0, 0.0, 10.2, 10.1],
    "high": [10.5, 10.4, 10.4, 10.4, 10.3],
    "low": [9.8, 10.0, 10.0, 10.1, 10.0],
    "close": [10.0, 10.2, 10.2, 10.3, 10.1],
    "adj_close": [9.0, 9.2, 9.2, np.nan, 9.1],
    "volume": [1000, 0, 0, 500, 700],
})
demo_prices = prepare_price_frame(demo_raw)
demo_prices["return"] = calculate_daily_returns(demo_prices)
demo_prices = (demo_prices.join(flag_return_outliers(demo_prices))
               .join(flag_ohlc_bounds(demo_prices))
               .join(flag_zero_volume(demo_prices)))
demo_calendar = pd.DataFrame({"ticker": "AAA",
                              "ts": pd.to_datetime(["2024-01-02", "2024-01-03", "2024-01-04", "2024-01-05", "2024-01-08"])})
demo_report = build_quality_report(demo_raw, demo_prices, find_missing_trading_days(demo_prices, demo_calendar))

assert list(demo_report.columns) == REPORT_COLS
assert set(demo_report["issue_type"]) == {"invalid_price", "ohlc_bounds", "missing_trading_day",
                                          "duplicate_record", "missing_value", "zero_volume"}
assert (demo_report["issue_date"] == "2024-01-03").sum() == 4  # zero open, bounds, duplicate, zero volume
assert demo_report.loc[demo_report["issue_type"] == "missing_trading_day", "issue_date"].tolist() == [pd.Timestamp("2024-01-04")]
demo_report

,ticker,issue_type,issue_date,description,severity
0,AAA,invalid_price,2024-01-03,nonpositive open (0.0),high
1,AAA,ohlc_bounds,2024-01-03,"open < low (O=0.0, H=10.4, L=10.0, C=10.2)",high
2,AAA,missing_trading_day,2024-01-04,date in reference calendar but no row,high
3,AAA,duplicate_record,2024-01-03,1 exact duplicate copy removed,medium
4,AAA,missing_value,2024-01-05,missing adj_close,medium
5,AAA,zero_volume,2024-01-03,volume = 0,low


#### Report on `ohlc_hw`

In [38]:
quality_report_before = build_quality_report(raw, prices, missing_days)
print(f"{len(quality_report_before):,} issues")
quality_report_before.head(20)

52,975 issues


,ticker,issue_type,issue_date,description,severity
0,CVX,invalid_price,1962-01-02,nonpositive open (0.0),high
1,CVX,ohlc_bounds,1962-01-02,"open < low (O=0.0, H=3.3, L=3.24, C=3.3)",high
2,JNJ,invalid_price,1962-01-02,nonpositive open (0.0),high
3,JNJ,ohlc_bounds,1962-01-02,"open < low (O=0.0, H=0.22338, L=0.22222, C=0.2...",high
4,MCD,invalid_price,1966-07-05,nonpositive open (0.0),high
5,MCD,ohlc_bounds,1966-07-05,"open < low (O=0.0, H=0.27366, L=0.26749, C=0.2...",high
6,MRK,invalid_price,1962-01-02,nonpositive open (0.0),high
7,MRK,ohlc_bounds,1962-01-02,"open < low (O=0.0, H=0.39703, L=0.39317, C=0.3...",high
8,PG,invalid_price,1962-01-02,nonpositive open (0.0),high
9,PG,ohlc_bounds,1962-01-02,"open < low (O=0.0, H=1.44, L=1.42, C=1.43)",high


Summary by issue type and severity:

In [39]:
pd.crosstab(quality_report_before["issue_type"], quality_report_before["severity"], margins=True, margins_name="total")

severity,high,medium,low,total
issue_type,,,,
duplicate_record,0,29460,0,29460
invalid_price,5,0,0,5
missing_value,0,18374,0,18374
ohlc_bounds,5,0,0,5
return_outlier,0,63,4993,5056
zero_volume,0,0,75,75
total,10,47897,5068,52975


Issues by ticker:

In [40]:
pd.crosstab(quality_report_before["ticker"], quality_report_before["issue_type"])

issue_type,duplicate_record,invalid_price,missing_value,ohlc_bounds,return_outlier,zero_volume
ticker,,,,,,
AAPL,892,0,575,0,143,0
AMGN,869,0,492,0,172,0
AMZN,521,0,328,0,147,0
AXP,1008,0,644,0,199,1
BA,1196,0,820,0,210,0
CAT,1236,0,831,0,221,0
CRM,408,0,250,0,82,0
CSCO,693,0,433,0,148,0
CVX,1230,1,817,1,178,0


The issue types the detectors checked for but did **not** find in this data are reported explicitly, so a zero is a
finding rather than an omission:

In [41]:
checked_types = ["missing_value", "invalid_price", "missing_trading_day", "duplicate_record",
                 "conflicting_record", "ohlc_bounds", "return_outlier", "zero_volume"]
quality_report_before["issue_type"].value_counts().reindex(checked_types, fill_value=0).rename("issues").to_frame()

,issues
issue_type,
missing_value,18374
invalid_price,5
missing_trading_day,0
duplicate_record,29460
conflicting_record,0
ohlc_bounds,5
return_outlier,5056
zero_volume,75


## 1.3 Clean the Data

Cleaning works on `clean`, a copy of the de-duplicated, sorted `prices` frame. `raw` and the database table
`ohlc_hw` are never modified. Rows that cannot be repaired defensibly are logged in `quarantine` with a reason,
not silently dropped. Each repair adds a boolean flag column, so every changed value can be traced.

Order of the missing-value steps:
1. drop rows where **all** raw prices are missing,
2. forward-fill remaining missing raw prices,
3. fill missing volume,
4. rebuild missing `adj_close`.

In [42]:
RAW_PRICE_COLS = ["open", "high", "low", "close"]

quarantine = pd.DataFrame(columns=["ticker", "ts", "reason"])


def quarantine_rows(df: pd.DataFrame, mask: pd.Series, reason: str) -> pd.DataFrame:
    """Return the ``ticker``/``ts`` of rows in ``df`` selected by ``mask``, labelled with ``reason``."""
    return df.loc[mask, ["ticker", "ts"]].assign(reason=reason)


clean = prices[["ticker_raw", "ticker", "ts"] + PRICE_COLS + ["volume"]].copy()
print(f"Rows in clean: {len(clean):,}")

Rows in clean: 368,251


### Drop rows where all raw prices are missing

A row with no `open`, `high`, `low`, or `close` has no price information at all. It must be removed **before**
forward-filling. If it were filled first, the fill would copy the whole previous bar into it, creating a day that
looks complete but is invented, and nothing in the data would show it was ever empty.

Only rows missing **all four** raw prices are dropped. A row missing some of them still has real information and goes
to the forward-fill step. `adj_close` and `volume` are not part of the test: a row with only an adjusted close and
no raw prices is still dropped, because adjusted close is derived from raw close and cannot stand alone.

Dropped rows are logged in `quarantine` with the reason, so the row accounting at the end is complete.

In [43]:
def drop_all_missing_prices(df: pd.DataFrame, cols: list[str] = RAW_PRICE_COLS) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Split ``df`` into rows with at least one raw price and rows with none.

    Returns ``(kept, dropped)`` where ``dropped`` is in quarantine format.
    """
    all_missing = df[cols].isna().all(axis=1)
    return df[~all_missing].copy(), quarantine_rows(df, all_missing, "all raw prices missing")

Check on an injected example: one complete row, one with all four raw prices missing (but an `adj_close` and volume),
and one with only some prices missing, which must be kept. The last assertion shows why the order matters:
filling first would erase all evidence that the row was empty.

In [44]:
demo = pd.DataFrame({
    "ticker": ["AAA"] * 3,
    "ts": pd.to_datetime(["2024-01-02", "2024-01-03", "2024-01-04"]),
    "open":      [10.0, np.nan, np.nan],
    "high":      [10.5, np.nan,   10.4],
    "low":       [ 9.8, np.nan,   10.0],
    "close":     [10.2, np.nan,   10.3],
    "adj_close": [ 9.0,    9.1,    9.2],
    "volume":    [1000,    800,    900],
})
kept, dropped = drop_all_missing_prices(demo)

assert dropped["ts"].tolist() == [pd.Timestamp("2024-01-03")]   # all four missing -> dropped
assert kept["ts"].tolist() == pd.to_datetime(["2024-01-02", "2024-01-04"]).tolist()  # partial row kept

# Wrong order: forward-fill first and the empty row becomes a copy of 01-02, undetectable afterwards.
filled_first = demo.assign(**{c: demo.groupby("ticker")[c].ffill() for c in RAW_PRICE_COLS})
assert drop_all_missing_prices(filled_first)[1].empty

display(kept)
dropped

,ticker,ts,open,high,low,close,adj_close,volume
0,AAA,2024-01-02,10.0,10.5,9.8,10.2,9.0,1000
2,AAA,2024-01-04,NaN,10.4,10.0,10.3,9.2,900


,ticker,ts,reason
1,AAA,2024-01-03,all raw prices missing


#### Apply to `ohlc_hw`

In [45]:
clean, dropped = drop_all_missing_prices(clean)
quarantine = pd.concat([quarantine, dropped], ignore_index=True)

print(f"Rows with all raw prices missing: {len(dropped)}")
print(f"Rows with some (not all) raw prices missing: {clean[RAW_PRICE_COLS].isna().any(axis=1).sum()}")
print(f"Rows in clean: {len(clean):,}")

Rows with all raw prices missing: 0
Rows with some (not all) raw prices missing: 0
Rows in clean: 368,251


**Result.** No row in `ohlc_hw` is missing all four raw prices (in fact none is missing any), so nothing is dropped.
The step stays in the pipeline so the cleaning would still be correct on a dirtier delivery of the same table.

### Handle missing values: forward-fill raw prices

Rule: a missing raw price (`open`, `high`, `low`, `close`) takes the ticker's last known value for that column.
This runs **after** the all-missing rows are dropped above.

Details that make this safe:
1. **Fill within each ticker** (data is sorted by ticker and date), so a price never carries over from another stock.
2. **Leading gaps are left missing.** A ticker's first rows have no earlier price, so there is nothing to carry
   forward. Those rows are quarantined rather than filled backward from the future.
3. **Bounds are rechecked after filling.** A carried-forward price can fall outside today's real high/low (e.g.
   yesterday's close of 105 when today's range is 98–101). Such rows are quarantined, because the fill produced an
   impossible bar.
4. **`adj_close` is not forward-filled.** It is the raw close times an adjustment factor, and the factor changes
   whenever a split or dividend happens. Carrying it forward would be wrong on exactly those days. Missing
   `adj_close` is handled separately, using the reference table.

Each filled cell is marked in a `{column}_ffilled` flag column.

In [46]:
def forward_fill_prices(df: pd.DataFrame, cols: list[str] = RAW_PRICE_COLS) -> pd.DataFrame:
    """Forward-fill missing ``cols`` within each ticker and flag each filled cell.

    ``df`` must be sorted by ticker and date. Adds a ``{col}_ffilled`` boolean column
    per price column. Leading gaps (no earlier price) stay missing.
    """
    out = df.copy()
    for col in cols:
        filled = out.groupby("ticker")[col].ffill()
        out[f"{col}_ffilled"] = out[col].isna() & filled.notna()
        out[col] = filled
    return out


def post_fill_problems(df: pd.DataFrame, cols: list[str] = RAW_PRICE_COLS) -> pd.DataFrame:
    """Quarantine rows still missing a raw price, or whose filled bar breaks OHLC bounds."""
    still_missing = df[cols].isna().any(axis=1)
    any_filled = df[[f"{c}_ffilled" for c in cols]].any(axis=1)
    bad_bounds = any_filled & flag_ohlc_bounds(df)["ohlc_inconsistent"].fillna(False)
    return pd.concat([
        quarantine_rows(df, still_missing, "raw price missing with no earlier value (leading gap)"),
        quarantine_rows(df, bad_bounds, "forward-filled price breaks OHLC bounds"),
    ])

Check on an injected example (the real data has no missing raw prices — see below), covering a normal fill, a
leading gap, fills that would break today's range, and a second ticker that must not inherit the first ticker's
last price:

In [47]:
demo = pd.DataFrame({
    "ticker": ["AAA"] * 4 + ["BBB"] * 2,
    "ts": pd.to_datetime(["2024-01-02", "2024-01-04", "2024-01-05", "2024-01-08",
                          "2024-01-02", "2024-01-03"]),
    "open":  [10.0, 10.1, np.nan,  10.0, np.nan, 20.0],
    "high":  [10.5, 10.4,   9.9,   10.3,   20.5, 20.6],
    "low":   [ 9.8, 10.0,   9.5,    9.9,   19.8, 19.9],
    "close": [10.2, 10.3,   9.6, np.nan,   20.1, 20.2],
})
#  AAA 01-05: open filled from 10.1 -> above today's high 9.9 -> quarantined
#  AAA 01-08: close filled from 9.6 -> below today's low 9.9  -> quarantined
#  BBB 01-02: leading gap in open   -> stays NaN (not AAA's 10.1) -> quarantined

filled = forward_fill_prices(demo)
problems = post_fill_problems(filled)

assert pd.isna(filled.loc[filled["ticker"] == "BBB", "open"].iloc[0])
assert filled.loc[filled["ts"] == "2024-01-05", "open"].iloc[0] == 10.1
assert set(zip(problems["ticker"], problems["ts"].dt.strftime("%m-%d"))) == {("AAA", "01-05"), ("AAA", "01-08"), ("BBB", "01-02")}

display(filled)
problems

,ticker,ts,open,high,low,close,open_ffilled,high_ffilled,low_ffilled,close_ffilled
0,AAA,2024-01-02,10.0,10.5,9.8,10.2,False,False,False,False
1,AAA,2024-01-04,10.1,10.4,10.0,10.3,False,False,False,False
2,AAA,2024-01-05,10.1,9.9,9.5,9.6,True,False,False,False
3,AAA,2024-01-08,10.0,10.3,9.9,9.6,False,False,False,True
4,BBB,2024-01-02,NaN,20.5,19.8,20.1,False,False,False,False
5,BBB,2024-01-03,20.0,20.6,19.9,20.2,False,False,False,False


,ticker,ts,reason
4,BBB,2024-01-02,raw price missing with no earlier value (leadi...
2,AAA,2024-01-05,forward-filled price breaks OHLC bounds
3,AAA,2024-01-08,forward-filled price breaks OHLC bounds


#### Apply to `ohlc_hw`

In [48]:
print("Missing raw prices before filling:")
print(clean[RAW_PRICE_COLS].isna().sum().to_string())

clean = forward_fill_prices(clean)
problems = post_fill_problems(clean)
quarantine = pd.concat([quarantine, problems], ignore_index=True)

print("\nCells forward-filled:")
print(clean[[f"{c}_ffilled" for c in RAW_PRICE_COLS]].sum().to_string())
print(f"Rows quarantined after filling: {len(problems)}")
print(f"\nRows in clean: {len(clean):,}")

Missing raw prices before filling:
open     0
high     0
low      0
close    0

Cells forward-filled:
open_ffilled     0
high_ffilled     0
low_ffilled      0
close_ffilled    0
Rows quarantined after filling: 0

Rows in clean: 368,251


**Result.** `ohlc_hw` has **no missing raw prices**, so nothing is filled or quarantined at this step; the
`*_ffilled` flags are all `False` and are kept so the cleaned table documents that. The forward-fill path is still
exercised by the injected example above.

The only missing prices in the data are the 18,374 `adj_close` values, which this step deliberately leaves alone.
The 5 zero opens are *invalid*, not missing, and are handled with the impossible-price repairs.

### Handle missing values: volume

Rule: a missing `volume` is filled with `0` **and flagged** in `volume_filled`.

The flag matters because a filled 0 and a reported 0 mean different things:
- **reported 0** (`volume_filled = False`): the vendor says no shares traded — evidence of a stale, no-trade day;
- **filled 0** (`volume_filled = True`): we don't know how many shares traded. Unknown volume is **not** evidence of no
  trading, so these rows must not be counted as zero-volume / stale days, and any volume-based statistic (average
  volume, liquidity screens) should exclude them.

Missing volume does not affect prices, so these rows stay in the data. Volume is a share count, so after filling it is
stored as an integer again (a column with `NaN` would otherwise stay `float`).

In [49]:
def fill_missing_volume(df: pd.DataFrame) -> pd.DataFrame:
    """Fill missing ``volume`` with 0, add a ``volume_filled`` flag, and store volume as int64.

    Raises if any non-missing volume is negative or non-finite, since those cannot be stored as share counts.
    """
    out = df.copy()
    v = out["volume"].astype(float)
    bad = v.notna() & (~np.isfinite(v) | (v < 0))
    if bad.any():
        raise ValueError(f"{bad.sum()} rows have negative or non-finite volume - investigate before filling")

    out["volume_filled"] = v.isna()
    out["volume"] = v.fillna(0).astype("int64")
    return out


def stale_zero_volume(df: pd.DataFrame) -> pd.Series:
    """Zero-volume days that the vendor actually reported (excludes volumes we filled)."""
    return (df["volume"] == 0) & ~df["volume_filled"]

Check on an injected example: one reported zero, one missing volume, and normal days:

In [50]:
demo = pd.DataFrame({
    "ticker": ["AAA"] * 4,
    "ts": pd.to_datetime(["2024-01-02", "2024-01-03", "2024-01-04", "2024-01-05"]),
    "volume": [1200, np.nan, 0, 900],
})
demo = fill_missing_volume(demo)

assert demo["volume"].tolist() == [1200, 0, 0, 900] and demo["volume"].dtype == "int64"
assert demo["volume_filled"].tolist() == [False, True, False, False]
assert stale_zero_volume(demo).tolist() == [False, False, True, False]  # only the reported zero counts
demo.assign(stale_zero_volume=stale_zero_volume(demo))

,ticker,ts,volume,volume_filled,stale_zero_volume
0,AAA,2024-01-02,1200,False,False
1,AAA,2024-01-03,0,True,False
2,AAA,2024-01-04,0,False,True
3,AAA,2024-01-05,900,False,False


#### Apply to `ohlc_hw`

In [51]:
print(f"Missing volume before filling: {clean['volume'].isna().sum()}")
clean = fill_missing_volume(clean)

print(f"Volumes filled with 0:          {int(clean['volume_filled'].sum())}")
print(f"Reported zero-volume days:      {int(stale_zero_volume(clean).sum())}")
print(f"volume dtype: {clean['volume'].dtype}, missing after: {clean['volume'].isna().sum()}")

Missing volume before filling: 0
Volumes filled with 0:          0
Reported zero-volume days:      75
volume dtype: int64, missing after: 0


**Result.** `ohlc_hw` has **no missing volume**, so no values are filled; `volume_filled` is all `False` and is saved
with the cleaned data. The 75 zero-volume days found earlier are all *reported* zeros, so they remain low-severity
stale-data flags, unaffected by this step.

### Handle missing values: adjusted close

**Convention.** `adj_close = close × adj_factor`, where `adj_factor` is the cumulative split-and-dividend adjustment
**for that specific date**. The factor changes on every split and dividend, so it cannot be forward-filled: carrying
yesterday's factor across an ex-dividend or split date would silently mis-scale every earlier price.

**Source of the factor.** The supplied `ohlc` table (same database) is used as the reference. For each `(ticker, ts)`:
`adj_factor = ohlc.adj_close / ohlc.close`. It is used only after checking that:
- the row matches exactly one reference row on `(ticker, ts)`, and
- the reference raw OHLC and volume agree with our row (so the factor belongs to the same bar), and
- the factor is finite and positive.

**Reconstruction.** `adj_close = validated raw close × adj_factor`. "Validated" means our close is present, finite,
positive, and within `[low, high]`. Rows that fail any check are **quarantined, not guessed**. Rows with no matched
reference factor are also quarantined; nothing is forward- or back-filled.

**Scope.** Only `adj_close` values are rebuilt, and only on rows that need it. The reference table never replaces our
raw prices or volume, and never adds or removes rows.

Columns added for traceability:
- `adj_close_vendor`: the original `ohlc_hw` value (NaN if it was missing)
- `adj_factor`: the reference factor used to check or rebuild the row
- `adj_close_source`: `vendor` (kept), `rebuilt_missing`, `rebuilt_inconsistent` (see below), or `unresolved` (quarantined)

In [52]:
RAW_MATCH_COLS = ["open", "high", "low", "close", "volume"]


def load_reference_factors(db_path: Path) -> pd.DataFrame:
    """Load date-specific adjustment factors from the supplied ``ohlc`` reference table.

    Returns one row per (ticker, ts) with the reference raw columns (suffixed ``_ref``),
    ``adj_close_ref``, and ``adj_factor = adj_close_ref / close_ref`` (NaN unless finite and positive).
    """
    with sqlite3.connect(db_path) as conn:
        ref = pd.read_sql_query("SELECT ticker, ts, open, high, low, close, adj_close, volume FROM ohlc", conn)
    ref["ts"] = pd.to_datetime(ref["ts"], format="%Y-%m-%d")
    ref = ref.rename(columns={c: f"{c}_ref" for c in RAW_MATCH_COLS + ["adj_close"]})

    if ref.duplicated(["ticker", "ts"]).any():
        raise ValueError("reference ohlc has duplicate (ticker, ts) keys - factors would be ambiguous")

    factor = ref["adj_close_ref"] / ref["close_ref"]
    ref["adj_factor"] = factor.where(np.isfinite(factor) & (factor > 0))
    return ref


def reconcile_adj_close(df: pd.DataFrame, ref: pd.DataFrame, rtol: float = 1e-6) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Rebuild missing or inconsistent ``adj_close`` as ``close * adj_factor`` from matched reference rows.

    A vendor ``adj_close`` is *consistent* when it equals ``close * adj_factor`` within ``rtol``.
    Returns ``(out, unresolved)``; ``unresolved`` lists, in quarantine format, rows whose adj_close is
    missing or inconsistent but cannot be rebuilt.
    """
    out = df.merge(ref, on=["ticker", "ts"], how="left", validate="one_to_one")

    close_valid = (np.isfinite(out["close"]) & (out["close"] > 0)
                   & (out["close"] >= out["low"]) & (out["close"] <= out["high"]))
    ref_matches_bar = np.logical_and.reduce([out[c] == out[f"{c}_ref"] for c in RAW_MATCH_COLS])
    usable = close_valid & ref_matches_bar & out["adj_factor"].notna()
    expected = out["close"] * out["adj_factor"]

    vendor = out["adj_close"]
    missing = vendor.isna()
    inconsistent = ~missing & usable & ~np.isclose(vendor, expected, rtol=rtol, atol=0)
    rebuild = (missing | inconsistent) & usable

    out["adj_close_vendor"] = vendor
    out["adj_close"] = vendor.where(~rebuild, expected)
    out["adj_close_source"] = np.select(
        [missing & rebuild, inconsistent, missing], ["rebuilt_missing", "rebuilt_inconsistent", "unresolved"],
        default="vendor",
    )

    reasons = np.select(
        [out["adj_factor"].isna() & out["close_ref"].isna(), out["adj_factor"].isna(), ~ref_matches_bar, ~close_valid],
        ["no matching reference row", "reference factor not finite/positive",
         "reference bar differs from ours", "raw close not valid"],
        default="",
    )
    unresolved = out.loc[missing & ~usable, ["ticker", "ts"]].assign(
        reason="adj_close missing, cannot rebuild: " + pd.Series(reasons, index=out.index)[missing & ~usable]
    )

    out = out.drop(columns=[f"{c}_ref" for c in RAW_MATCH_COLS + ["adj_close"]])
    return out, unresolved

Check on an injected example covering every path, including the "no forward-fill" rule: on 01-05 the factor
changes (an ex-dividend date), and 01-08 has no reference row. Forward-filling would give 01-08 the 01-05 factor;
instead it must be quarantined.

In [53]:
demo_ref = pd.DataFrame({
    "ticker": "AAA",
    "ts": pd.to_datetime(["2024-01-02", "2024-01-03", "2024-01-04", "2024-01-05", "2024-01-09", "2024-01-10"]),
    "open_ref": 10.0, "high_ref": 11.0, "low_ref": 9.0, "close_ref": 10.0, "volume_ref": 100,
    "adj_close_ref": [9.0, 9.0, 9.0, 9.5, 9.5, -1.0],
})
demo_ref.loc[demo_ref["ts"] == "2024-01-09", "close_ref"] = 10.4  # reference bar disagrees with ours
demo_ref["adj_factor"] = (demo_ref["adj_close_ref"] / demo_ref["close_ref"]).where(lambda f: f > 0)

demo = pd.DataFrame({
    "ticker": "AAA",
    "ts": pd.to_datetime(["2024-01-02", "2024-01-03", "2024-01-04", "2024-01-05",
                          "2024-01-08", "2024-01-09", "2024-01-10"]),
    "open": 10.0, "high": 11.0, "low": 9.0, "close": 10.0, "volume": 100,
    "adj_close": [9.0, np.nan, 135.0, np.nan, np.nan, np.nan, np.nan],
})
#  01-02 consistent vendor value -> kept
#  01-03 missing                 -> rebuilt 10.0 * 0.90 = 9.0
#  01-04 vendor value 15x too big -> rebuilt to 9.0
#  01-05 missing, new factor     -> rebuilt 10.0 * 0.95 = 9.5
#  01-08 missing, no ref row     -> quarantined (NOT 9.5 by forward-fill)
#  01-09 ref close 10.4 != ours  -> quarantined
#  01-10 ref factor negative     -> quarantined
demo_out, demo_unresolved = reconcile_adj_close(demo, demo_ref)

assert demo_out["adj_close_source"].tolist()[:4] == ["vendor", "rebuilt_missing", "rebuilt_inconsistent", "rebuilt_missing"]
assert np.allclose(demo_out["adj_close"].iloc[:4], [9.0, 9.0, 9.0, 9.5])
assert demo_out["adj_close"].iloc[4:].isna().all()
assert (demo_out["adj_close_source"].iloc[4:] == "unresolved").all()
assert demo_unresolved["ts"].dt.strftime("%m-%d").tolist() == ["01-08", "01-09", "01-10"]

display(demo_out[["ts", "close", "adj_close_vendor", "adj_factor", "adj_close", "adj_close_source"]])
demo_unresolved

,ts,close,adj_close_vendor,adj_factor,adj_close,adj_close_source
0,2024-01-02,10.0,9.0,0.900000,9.0,vendor
1,2024-01-03,10.0,NaN,0.900000,9.0,rebuilt_missing
2,2024-01-04,10.0,135.0,0.900000,9.0,rebuilt_inconsistent
3,2024-01-05,10.0,NaN,0.950000,9.5,rebuilt_missing
4,2024-01-08,10.0,NaN,NaN,NaN,unresolved
5,2024-01-09,10.0,NaN,0.913462,NaN,unresolved
6,2024-01-10,10.0,NaN,NaN,NaN,unresolved


,ticker,ts,reason
4,AAA,2024-01-08,"adj_close missing, cannot rebuild: no matching..."
5,AAA,2024-01-09,"adj_close missing, cannot rebuild: reference b..."
6,AAA,2024-01-10,"adj_close missing, cannot rebuild: reference f..."


#### Validate the reference source

Before trusting `ohlc` for factors, confirm it describes the same bars as our data: every row should match one
reference row, and the raw columns should agree.

In [54]:
ref_factors = load_reference_factors(DB_PATH)
src_check = clean[["ticker", "ts"] + RAW_MATCH_COLS].merge(
    ref_factors, on=["ticker", "ts"], how="left", validate="one_to_one", indicator=True
)

print(f"Rows in clean:            {len(clean):,}")
print(f"Matched to ohlc:          {(src_check['_merge'] == 'both').sum():,}")
for c in RAW_MATCH_COLS:
    print(f"  {c:6} differs from ref: {(src_check[c] != src_check[f'{c}_ref']).sum()}")
print(f"Usable reference factors: {src_check['adj_factor'].notna().sum():,}")
print(f"Factor range: {src_check['adj_factor'].min():.4f} to {src_check['adj_factor'].max():.4f}")

Rows in clean:            368,251
Matched to ohlc:          368,251
  open   differs from ref: 0
  high   differs from ref: 0
  low    differs from ref: 0
  close  differs from ref: 0
  volume differs from ref: 0
Usable reference factors: 368,251
Factor range: 0.1017 to 1.0000


Every row matches, our raw prices and volume are identical to the reference, and every factor is usable. So the
reference supplies a trustworthy, date-specific factor for every row.

#### A second adjusted-close problem: inconsistent vendor values

Comparing the vendor `adj_close` with `close × adj_factor` also reveals values that are **present but wrong**.
None of the checks in 1.2 caught them: they are positive and finite, and raw OHLC bounds deliberately do not apply to
adjusted close. Their pattern:

In [55]:
present = clean["adj_close"].notna().to_numpy()
ratio = (clean["adj_close"].to_numpy() / (src_check["close"] * src_check["adj_factor"]).to_numpy())
off = present & ~np.isclose(ratio, 1, rtol=1e-6)

print(f"Vendor adj_close present:              {present.sum():,}")
print(f"  consistent with close x factor:      {(present & ~off).sum():,}")
print(f"  inconsistent:                        {off.sum():,} ({off.sum() / present.sum():.2%})")
print(f"  vendor / expected, inconsistent rows: {ratio[off].min():.2f}x to {ratio[off].max():.2f}x "
      f"(median {np.median(ratio[off]):.2f}x)")
print(f"  share above raw high, inconsistent:  {(clean['adj_close'] > clean['high'])[off].mean():.1%}")
print(f"  share above raw high, consistent:    {(clean['adj_close'] > clean['high'])[present & ~off].mean():.4%}")
print()
print("Inconsistent share by ticker:")
print(pd.Series(off[present], index=clean.loc[present, "ticker"]).groupby(level=0).mean()
      .describe()[["min", "50%", "max"]].map("{:.2%}".format).to_string())

Vendor adj_close present:              349,877
  consistent with close x factor:      339,460
  inconsistent:                        10,417 (2.98%)
  vendor / expected, inconsistent rows: 4.00x to 24.00x (median 14.06x)
  share above raw high, inconsistent:  99.3%
  share above raw high, consistent:    0.0003%

Inconsistent share by ticker:
min    2.48%
50%    2.98%
max    3.39%


Every inconsistent value is 4x–24x too large, they are spread evenly (~3%) over every ticker, and nearly all sit
above the day's raw high, which consistent values essentially never do. There is no exact match or small rounding
difference in between: values are either exact or wildly off. This is a corrupted value, not a different adjustment
convention, so these values are rebuilt with the same factor as the missing ones and labelled
`rebuilt_inconsistent`. The original value stays in `adj_close_vendor`.

#### Apply to `ohlc_hw`

In [56]:
missing_before = clean["adj_close"].isna().sum()
clean, adj_unresolved = reconcile_adj_close(clean, ref_factors)
quarantine = pd.concat([quarantine, adj_unresolved], ignore_index=True)

print(f"adj_close missing before:           {missing_before:,}")
print(clean["adj_close_source"].value_counts().rename("rows").to_string())
print(f"Unresolved (quarantined):           {len(adj_unresolved):,}")
if len(adj_unresolved):
    print(adj_unresolved["reason"].value_counts().to_string())

adj_close missing before:           18,374
adj_close_source
vendor                  339460
rebuilt_missing          18374
rebuilt_inconsistent     10417
Unresolved (quarantined):           0


Recheck after the repair: every adjusted close outside quarantine must be present, finite, positive, and equal to `close × adj_factor`.

In [57]:
in_quarantine = clean.set_index(["ticker", "ts"]).index.isin(quarantine.set_index(["ticker", "ts"]).index)
kept = clean[~in_quarantine]

assert kept["adj_close"].notna().all(), "missing adj_close outside quarantine"
assert (np.isfinite(kept["adj_close"]) & (kept["adj_close"] > 0)).all(), "nonpositive or non-finite adj_close"
assert np.allclose(kept["adj_close"], kept["close"] * kept["adj_factor"], rtol=1e-6, atol=0), "adj_close inconsistent"
print(f"OK: {len(kept):,} rows have a present, positive adj_close consistent with close x factor")

clean[clean["adj_close_source"] != "vendor"][
    ["ticker", "ts", "close", "adj_close_vendor", "adj_factor", "adj_close", "adj_close_source"]
].head(10)

OK: 368,251 rows have a present, positive adj_close consistent with close x factor


,ticker,ts,close,adj_close_vendor,adj_factor,adj_close,adj_close_source
17,AAPL,1981-01-08,0.13505,NaN,0.770826,0.104100,rebuilt_missing
20,AAPL,1981-01-13,0.13616,NaN,0.770858,0.104960,rebuilt_missing
35,AAPL,1981-02-03,0.12333,1.264587,0.770853,0.095069,rebuilt_inconsistent
42,AAPL,1981-02-12,0.11663,NaN,0.770872,0.089907,rebuilt_missing
86,AAPL,1981-04-16,0.11161,NaN,0.770858,0.086035,rebuilt_missing
93,AAPL,1981-04-28,0.12612,2.036653,0.770854,0.097220,rebuilt_inconsistent
101,AAPL,1981-05-08,0.12500,2.129290,0.770878,0.096360,rebuilt_inconsistent
108,AAPL,1981-05-19,0.12277,NaN,0.770866,0.094639,rebuilt_missing
119,AAPL,1981-06-04,0.14341,NaN,0.770936,0.110560,rebuilt_missing
146,AAPL,1981-07-14,0.10603,NaN,0.770857,0.081734,rebuilt_missing


**Result.**
- **18,374 missing** `adj_close` values were rebuilt from validated raw close and the matched reference factor.
- **10,417 inconsistent** vendor values (4x–24x too large) were rebuilt the same way. This issue was found here,
  not in the 1.2 report, and it is a high-severity issue: it would wreck every adjusted-return calculation in Part 2.
- **0 rows unresolved**, because the reference covers every row. The quarantine path is still exercised in the example.
- No factor was forward-filled; every rebuilt value uses its own date's factor.

### Handle outliers: impossible price relationships

The rows to repair are those where one raw price breaks the OHLC bounds (found in 1.2: the 5 zero opens, each
below the day's low). Three candidate repairs for a bad price `X`:

| option | repair | needs |
|---|---|---|
| **1. NaN, then forward-fill** | `X` = the ticker's previous value of the *same* column (yesterday's open for a bad open) | a previous row |
| **2. Previous day's close** | `X` = the ticker's previous close | a previous row |
| **3. Interpolate from same-day prices** | `X` = midpoint of the day's range, `(high + low) / 2` | valid high and low today |

For option 3, the midpoint is used rather than the mean of all other prices: it is always inside `[low, high]`, and it
does not depend on the other price that might also be off.

**Which price is wrong?** Before repairing, the bad price must be identified. If the open is outside `[low, high]` but
the close is inside, the open is the outlier (and vice versa). If the range itself is inverted (`low > high`), or
both open and close are outside it, the bad value is ambiguous, so the row is quarantined rather than guessed.

In [58]:
def identify_bad_price(df: pd.DataFrame) -> pd.Series:
    """Name the single raw price breaking OHLC bounds: ``"open"``, ``"close"``, ``"ambiguous"``, or ``""`` (none)."""
    f = flag_ohlc_bounds(df).fillna(False)
    open_bad = f["open_below_low"] | f["open_above_high"]
    close_bad = f["close_below_low"] | f["close_above_high"]
    return pd.Series(
        np.select(
            [f["low_above_high"] | (open_bad & close_bad), open_bad, close_bad],
            ["ambiguous", "open", "close"],
            default="",
        ),
        index=df.index,
    )


def repair_ffill(df: pd.DataFrame, col: str) -> pd.Series:
    """Option 1: previous value of the same column, within each ticker."""
    return df.groupby("ticker")[col].shift(1)


def repair_prev_close(df: pd.DataFrame, col: str) -> pd.Series:
    """Option 2: previous day's close, within each ticker."""
    return df.groupby("ticker")["close"].shift(1)


def repair_interpolate(df: pd.DataFrame, col: str) -> pd.Series:
    """Option 3: midpoint of the same day's range."""
    return (df["high"] + df["low"]) / 2


REPAIR_OPTIONS = {
    "1: ffill same column": repair_ffill,
    "2: previous close": repair_prev_close,
    "3: midpoint (H+L)/2": repair_interpolate,
}

#### Comparing the options

The real data has only 5 impossible prices, and their true values are unknown (the reference table has the same zeros).
That is too little to compare methods on. Instead, the options are tested on **simulated** errors: take 20,000 random
valid bars, pretend one price is corrupted, repair it each way, and compare with the true value we hid. Two measures:
- **accuracy:** absolute % error vs. the true price (median and 90th percentile);
- **validity:** share of repaired bars that satisfy `low <= open, close <= high` (an invalid repair just swaps one error for another).

A fourth variant is included for the open: **option 2 clipped to the day's range**, i.e. `clip(prev_close, low, high)`. It
combines option 2's information (the open usually starts near yesterday's close) with option 3's guarantee of a valid bar.

In [59]:
def clip_to_range(est: pd.Series, df: pd.DataFrame) -> pd.Series:
    """Move an estimate into the day's [low, high] range."""
    return est.clip(lower=df["low"], upper=df["high"])


def compare_repairs(df: pd.DataFrame, col: str, n: int = 20_000, seed: int = 0) -> pd.DataFrame:
    """Score each repair option on ``n`` random valid bars where ``col`` is treated as corrupted.

    Bars are sampled from rows with a previous row in the same ticker and no bound violations.
    """
    candidates = df[
        df.groupby("ticker").cumcount().gt(0)
        & ~flag_ohlc_bounds(df)["ohlc_inconsistent"].fillna(True)
    ]
    sample_idx = candidates.sample(n, random_state=seed).index

    estimates = {name: fn(df, col) for name, fn in REPAIR_OPTIONS.items()}
    estimates["2 clipped: clip(prev close, L, H)"] = clip_to_range(repair_prev_close(df, col), df)

    truth, low, high = df.loc[sample_idx, col], df.loc[sample_idx, "low"], df.loc[sample_idx, "high"]
    rows = []
    for name, est in estimates.items():
        e = est.loc[sample_idx]
        ape = (e / truth - 1).abs()
        rows.append({
            "option": name,
            "median_abs_error": ape.median(),
            "p90_abs_error": ape.quantile(0.9),
            "valid_bar": ((e >= low) & (e <= high)).mean(),
        })
    return pd.DataFrame(rows).set_index("option")


fmt = {"median_abs_error": "{:.3%}", "p90_abs_error": "{:.3%}", "valid_bar": "{:.1%}"}
for col in ["open", "close"]:
    print(f"Simulated corrupted {col.upper()} (20,000 bars):")
    print(compare_repairs(clean, col).apply(lambda s: s.map(fmt[s.name].format)).to_string())
    print()

Simulated corrupted OPEN (20,000 bars):
                                  median_abs_error p90_abs_error valid_bar
option                                                                    
1: ffill same column                        0.893%        2.882%     46.3%
2: previous close                           0.256%        1.261%     72.5%
3: midpoint (H+L)/2                         0.504%        1.476%    100.0%
2 clipped: clip(prev close, L, H)           0.091%        0.826%    100.0%

Simulated corrupted CLOSE (20,000 bars):


                                  median_abs_error p90_abs_error valid_bar
option                                                                    
1: ffill same column                        0.916%        2.899%     72.5%
2: previous close                           0.916%        2.899%     72.5%
3: midpoint (H+L)/2                         0.459%        1.383%    100.0%
2 clipped: clip(prev close, L, H)           0.775%        2.526%    100.0%



Availability matters as much as accuracy. Options 1 and 2 need a previous row, so they cannot repair a
ticker's **first** day. That is exactly where all 5 real errors are:

In [60]:
bad_price = identify_bad_price(clean)
to_repair = clean[bad_price != ""]
is_first_day = clean.groupby("ticker").cumcount().eq(0)

print(f"Rows with an impossible price: {len(to_repair)}  ->  {bad_price[bad_price != ''].value_counts().to_dict()}")
print(f"  on the ticker's first day: {is_first_day[to_repair.index].sum()}")
print()
avail = pd.DataFrame({name: fn(clean, "open")[to_repair.index] for name, fn in REPAIR_OPTIONS.items()})
pd.concat([to_repair[["ticker", "ts", "open", "high", "low", "close"]], avail], axis=1)

Rows with an impossible price: 5  ->  {'open': 5}
  on the ticker's first day: 5



,ticker,ts,open,high,low,close,1: ffill same column,2: previous close,3: midpoint (H+L)/2
87103,CVX,1962-01-02,0.0,3.30000,3.24000,3.30000,NaN,NaN,3.270000
167733,JNJ,1962-01-02,0.0,0.22338,0.22222,0.22338,NaN,NaN,0.222800
213412,MCD,1966-07-05,0.0,0.27366,0.26749,0.26955,NaN,NaN,0.270575
243893,MRK,1962-01-02,0.0,0.39703,0.39317,0.39537,NaN,NaN,0.395100
287040,PG,1962-01-02,0.0,1.44000,1.42000,1.43000,NaN,NaN,1.430000


#### Decision

**Option 1 (forward-fill the same column) is the worst.** It is the least accurate for both columns and gives an
invalid bar about half the time when repairing the open. Yesterday's open has no particular reason to fall in today's range.

**Option 2 (previous close) is the best *information* for a bad open, but not for a bad close.** A stock usually opens
close to where it last closed, so its error for the open is about half of the midpoint's. (In the older data, the
vendor often recorded the open as the previous close, which makes it exact.) But it produces an invalid bar about a quarter of
the time, whenever the stock gapped up or down overnight. For a bad close it is poor: it ignores everything that
happened during the day.

**Option 3 (midpoint) is always available and always valid**, and it is the most accurate for a bad close, since the
close is somewhere in today's range and the midpoint is the center of that range.

**Chosen rule** (also the "document your choice" for the cleaning step):
- **bad open** → previous close clipped to the day's range: option 2's information, with option 3's guarantee of a valid
  bar. It had the lowest error of all variants. On a ticker's first day there is no previous close, so it **falls
  back to option 3** (the midpoint).
- **bad close** → option 3, the midpoint.
- **ambiguous** (inverted range, or both open and close out of range) → quarantine.

Every repaired cell is flagged and the method is recorded. The original value is kept in `{col}_vendor`.

In [61]:
def repair_impossible_prices(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Repair a single out-of-bounds open or close using the chosen rule.

    Bad close -> midpoint (high + low) / 2.
    Bad open  -> clip(previous close, low, high), or the midpoint on a ticker's first day.
    Closes are repaired first, so a bad open never borrows yesterday's *bad* close.
    Ambiguous rows are not changed and are returned in quarantine format.
    Adds ``open_vendor``/``close_vendor``, ``open_repaired``/``close_repaired`` and ``price_repair_method``.
    """
    out = df.copy()
    bad = identify_bad_price(out)
    midpoint = repair_interpolate(out, "close")
    out["open_vendor"], out["close_vendor"] = out["open"], out["close"]

    # 1) Closes first (option 3).
    out["close_repaired"] = bad == "close"
    out.loc[out["close_repaired"], "close"] = midpoint[out["close_repaired"]]

    # 2) Opens, using the already-repaired closes (option 2 clipped, option 3 fallback).
    prev_close = repair_prev_close(out, "open")
    out["open_repaired"] = bad == "open"
    open_est = clip_to_range(prev_close, out).fillna(midpoint)
    out.loc[out["open_repaired"], "open"] = open_est[out["open_repaired"]]

    out["price_repair_method"] = np.select(
        [out["close_repaired"], out["open_repaired"] & prev_close.notna(), out["open_repaired"]],
        ["close: midpoint", "open: clipped previous close", "open: midpoint (first day)"],
        default="",
    )
    return out, quarantine_rows(out, bad == "ambiguous", "ambiguous OHLC bound violation")

Check on injected bars, built so that each mistake would show up:
- `AAA 01-03`: close 12.0 is above the high, so close → midpoint 10.2.
- `AAA 01-04`: open is 0. Its previous close must be the **repaired** 10.2, clipped to today's low, so open = 10.2.
  If the bad close 12.0 were used, the result would be 10.6 instead.
- `AAA 01-05`: open and close both above the high → ambiguous → quarantined, unchanged.
- `BBB 01-02`: open is 0 on the first day → no previous close → midpoint 20.0 (and not AAA's last close).

In [62]:
demo = pd.DataFrame({
    "ticker": ["AAA"] * 4 + ["BBB"],
    "ts": pd.to_datetime(["2024-01-02", "2024-01-03", "2024-01-04", "2024-01-05", "2024-01-02"]),
    "open":  [10.0, 10.1,  0.0, 11.0,  0.0],
    "high":  [10.5, 10.4, 10.6, 10.5, 20.4],
    "low":   [ 9.8, 10.0, 10.2,  9.0, 19.6],
    "close": [10.2, 12.0, 10.5, 11.0, 20.1],
})
demo_out, demo_q = repair_impossible_prices(demo)

assert np.allclose(demo_out["close"], [10.2, 10.2, 10.5, 11.0, 20.1])
assert np.allclose(demo_out["open"], [10.0, 10.1, 10.2, 11.0, 20.0])
assert demo_q["ts"].dt.strftime("%m-%d").tolist() == ["01-05"]
display(demo_out[["ticker", "ts", "open_vendor", "open", "close_vendor", "close", "high", "low", "price_repair_method"]])
demo_q

,ticker,ts,open_vendor,open,close_vendor,close,high,low,price_repair_method
0,AAA,2024-01-02,10.0,10.0,10.2,10.2,10.5,9.8,
1,AAA,2024-01-03,10.1,10.1,12.0,10.2,10.4,10.0,close: midpoint
2,AAA,2024-01-04,0.0,10.2,10.5,10.5,10.6,10.2,open: clipped previous close
3,AAA,2024-01-05,11.0,11.0,11.0,11.0,10.5,9.0,
4,BBB,2024-01-02,0.0,20.0,20.1,20.1,20.4,19.6,open: midpoint (first day)


,ticker,ts,reason
3,AAA,2024-01-05,ambiguous OHLC bound violation


#### Apply to `ohlc_hw`

In [63]:
clean, price_q = repair_impossible_prices(clean)
quarantine = pd.concat([quarantine, price_q], ignore_index=True)

# A repaired close changes adjusted close too: keep adj_close = close x date-specific factor.
clean.loc[clean["close_repaired"], "adj_close"] = clean["close"] * clean["adj_factor"]

repaired = clean[clean["open_repaired"] | clean["close_repaired"]]
print(f"Open repaired: {int(clean['open_repaired'].sum())}, close repaired: {int(clean['close_repaired'].sum())}, "
      f"ambiguous (quarantined): {len(price_q)}")
print(repaired["price_repair_method"].value_counts().to_string())

# Recheck: no bound violations remain outside quarantine, and adj_close is still consistent.
in_q = clean.set_index(["ticker", "ts"]).index.isin(quarantine.set_index(["ticker", "ts"]).index)
assert not flag_ohlc_bounds(clean[~in_q])["ohlc_inconsistent"].any(), "bound violations remain"
assert (clean.loc[~in_q, RAW_PRICE_COLS] > 0).all().all(), "nonpositive raw price remains"
assert np.allclose(clean.loc[~in_q, "adj_close"], clean.loc[~in_q, "close"] * clean.loc[~in_q, "adj_factor"], rtol=1e-6)
print("OK: no OHLC bound violations or nonpositive raw prices remain; adj_close still consistent")

repaired[["ticker", "ts", "open_vendor", "open", "high", "low", "close", "price_repair_method"]]

Open repaired: 5, close repaired: 0, ambiguous (quarantined): 0
price_repair_method
open: midpoint (first day)    5


OK: no OHLC bound violations or nonpositive raw prices remain; adj_close still consistent


,ticker,ts,open_vendor,open,high,low,close,price_repair_method
87103,CVX,1962-01-02,0.0,3.270000,3.30000,3.24000,3.30000,open: midpoint (first day)
167733,JNJ,1962-01-02,0.0,0.222800,0.22338,0.22222,0.22338,open: midpoint (first day)
213412,MCD,1966-07-05,0.0,0.270575,0.27366,0.26749,0.26955,open: midpoint (first day)
243893,MRK,1962-01-02,0.0,0.395100,0.39703,0.39317,0.39537,open: midpoint (first day)
287040,PG,1962-01-02,0.0,1.430000,1.44000,1.42000,1.43000,open: midpoint (first day)


**Result.** All 5 impossible prices were zero opens on a ticker's first day, so all 5 used the fallback,
the midpoint of that day's range (option 3). Options 1 and 2 could not have repaired them at all. No close needed
repair, so `adj_close` is unchanged. Afterwards, no row breaks the OHLC bounds and no raw price is zero or negative.

The repaired opens are estimates, not recovered truths: the reference table has the same zeros, so the real
opens are unknown. They are flagged (`open_repaired`, `price_repair_method`), and the vendor zero is kept in
`open_vendor`. The open is not used for returns, volatility, or drawdowns, so this estimate has no effect on the Part 2 risk numbers.

### Summary: handling impossible price relationships

**What counted as an outlier here.** A raw price that breaks the bar's own bounds (`low <= open, close <= high`).
Such a price cannot be real, so it must be repaired or quarantined. Extreme *returns* (the 3-std screen) are a different
kind of outlier, since the price may be real. They are handled separately in the next step.

**Process.**
1. **Identify the bad price.** For each violating bar, decide whether the open or the close is the outlier. If the range
   itself is inverted, or both are out of range, the bad value is ambiguous and the row is quarantined.
2. **Compare the three assignment options** on 20,000 simulated corruptions of valid bars (the 5 real errors have
   unknown true values, so they cannot be used for scoring):

   | | bad open: median error | bad open: valid bar | bad close: median error | bad close: valid bar |
   |---|---|---|---|---|
   | 1: NaN → forward-fill | 0.89% | 46% | 0.92% | 73% |
   | 2: previous close | 0.26% | 73% | 0.92% | 73% |
   | 3: interpolate (midpoint of H, L) | 0.50% | 100% | **0.46%** | 100% |
   | 2 clipped to [L, H] | **0.09%** | 100% | 0.77% | 100% |

3. **Choose a rule per column**, based on accuracy, validity, and availability:
   - **bad open:** previous close clipped to the day's range (option 2's information plus option 3's validity guarantee).
     On a ticker's first day there is no previous close, so it falls back to option 3.
   - **bad close:** option 3, the midpoint of the day's range.
   - **ambiguous:** quarantine.
   - Closes are repaired before opens, so an open never borrows yesterday's bad close.
4. **Repair, flag, and recheck.** Originals are kept in `open_vendor` / `close_vendor`. Each repair is marked in
   `open_repaired` / `close_repaired` with its `price_repair_method`. A repaired close also updates
   `adj_close = close × adj_factor`. Afterwards all bounds, price positivity, and adjusted-close consistency are rechecked.

**Why not a single option for everything?** Option 1 is the least accurate and often leaves the bar still impossible.
Option 2 is the best information for an open but often lands outside today's range, and it is poor for a close. Option 3
always works and always gives a valid bar. Neither option 1 nor option 2 can repair a ticker's first day, which is
where all of this dataset's errors are.

**Final results** (computed live below):

In [64]:
outlier_summary = pd.DataFrame({
    "count": {
        "bars breaking OHLC bounds (before)": int(flag_ohlc_bounds(prices)["ohlc_inconsistent"].sum()),
        "  bad open": int((identify_bad_price(prices) == "open").sum()),
        "  bad close": int((identify_bad_price(prices) == "close").sum()),
        "  ambiguous -> quarantined": len(price_q),
        "opens repaired": int(clean["open_repaired"].sum()),
        "  via clipped previous close": int((clean["price_repair_method"] == "open: clipped previous close").sum()),
        "  via midpoint (first day)": int((clean["price_repair_method"] == "open: midpoint (first day)").sum()),
        "closes repaired (midpoint)": int(clean["close_repaired"].sum()),
        "bars breaking OHLC bounds (after, outside quarantine)":
            int(flag_ohlc_bounds(clean[~in_q])["ohlc_inconsistent"].sum()),
        "nonpositive raw prices (after, outside quarantine)":
            int((clean.loc[~in_q, RAW_PRICE_COLS] <= 0).any(axis=1).sum()),
    }
})
outlier_summary

,count
bars breaking OHLC bounds (before),5
bad open,5
bad close,0
ambiguous -> quarantined,0
opens repaired,5
via clipped previous close,0
via midpoint (first day),5
closes repaired (midpoint),0
"bars breaking OHLC bounds (after, outside quarantine)",0
"nonpositive raw prices (after, outside quarantine)",0


- All **5** impossible prices were **zero opens on each ticker's first trading day** (CVX, JNJ, MRK, PG on 1962-01-02;
  MCD on 1966-07-05). The reference table has the same zeros, so they are a vendor placeholder for an unrecorded open.
- All 5 were repaired with the **midpoint of that day's range** (the first-day fallback). No close needed repair and no
  row was ambiguous, so nothing was quarantined at this step and `adj_close` was unchanged.
- After repair, **no bar breaks the OHLC bounds and no raw price is zero or negative.**
- **Caveat:** these opens are estimates, not recovered values. They are flagged, and the vendor zero is preserved. The
  open is not used for returns, volatility, or drawdowns, so the Part 2 risk metrics are unaffected.

**Next:** the extreme-return outliers (5,056 review flags, 63 of them at |z| ≥ 10) are investigated in the following section.

### Handle outliers: extreme returns (> 3 std)

The 3-std screen from 1.2 flags 5,056 days. A flag only says the move was unusual. Each one has to be explained before
deciding what to do:

| verdict | what it means | action |
|---|---|---|
| `data_error` | the price is wrong for one day | interpolate from the neighboring days |
| `data_error_reversal` | the day *after* a data error, whose return is only extreme because of the bad day | fixed by repairing the error |
| `corporate_action` | a split/dividend: the raw price moved, but the adjustment factor changed to match | keep, flag |
| `series_break` | the price scale changes permanently and the vendor did not adjust it | quarantine (see below) |
| `market_event` | most stocks made extreme moves the same day | keep, flag |
| `genuine_unconfirmed` | no sign of an error; consistent with a real move | keep, flag |

**Evidence used**, all computed from the data itself:
- **Reversal:** does the next close return to the level before the move? A real move rarely undoes itself exactly the
  next day; a bad print does. Measured as `log(close[t+1] / close[t-1])` relative to the size of the move.
- **Volume:** the day's volume relative to its median over the previous 20 days. Real news trades heavily; a bad
  print on a no-trade day has no trading behind it.
- **Adjustment factor** (`adj_close / close`): a split or dividend changes the factor on its ex-date and the new factor
  persists. If the raw move is extreme but the *adjusted* move is not, the corporate action explains it.
- **Market breadth:** the share of all tickers trading that day that were flagged too.

**What the data cannot show.** There is no corporate-actions or news feed here, so a **merger** cannot be identified
directly. It would look like a large move with heavy volume, and would be labelled `genuine_unconfirmed`. Likewise
`genuine_unconfirmed` means "no evidence of an error", not "confirmed news". Specific news is never claimed.

In [65]:
def return_evidence(df: pd.DataFrame, vol_window: int = 20) -> pd.DataFrame:
    """Evidence for explaining extreme returns, aligned to ``df`` (sorted by ticker, ts).

    Columns: ``log_move`` (log close/prev close), ``reversal`` (log next close/prev close),
    ``vol_ratio`` (volume / median of previous ``vol_window`` days), ``factor_step``
    (adj factor change vs previous day), ``factor_holds`` (next day keeps the new factor),
    ``adj_return_outlier`` (3-std screen on adjusted close returns), ``market_share``
    (share of tickers trading that day with a raw-return flag).
    """
    g = df.groupby("ticker")
    prev_close, next_close = g["close"].shift(1), g["close"].shift(-1)
    factor = df["adj_factor"]

    adj = df[["ticker"]].assign(**{"return": calculate_daily_returns(df, "adj_close")})
    raw_flag = flag_return_outliers(df)["return_outlier"].fillna(False)

    return pd.DataFrame({
        "log_move": np.log(df["close"] / prev_close),
        "reversal": np.log(next_close / prev_close),
        "vol_ratio": df["volume"] / g["volume"].transform(
            lambda v: v.rolling(vol_window, min_periods=5).median().shift(1)),
        "factor_step": factor / g["adj_factor"].shift(1) - 1,
        "factor_holds": (g["adj_factor"].shift(-1) / factor - 1).abs() < 1e-3,
        "adj_return_outlier": flag_return_outliers(adj)["return_outlier"].fillna(False),
        "market_share": raw_flag.groupby(df["ts"]).transform("mean"),
    }, index=df.index)


def classify_extreme_returns(df: pd.DataFrame, evidence: pd.DataFrame, manual: dict | None = None) -> pd.Series:
    """Give each flagged return a verdict ("" for unflagged rows).

    Rules, in priority order:
      manual override (documented separately) >
      data_error: |log move| > 0.15, next day undoes >= 90% of it, volume under half its usual level,
                  and not a market-wide day >
      data_error_reversal: the previous day is a data_error >
      corporate_action: adjustment factor steps > 1% and holds, and the adjusted return is not extreme >
      market_event: >= 50% of tickers trading that day were flagged >
      genuine_unconfirmed.
    """
    flagged = df["return_outlier"].fillna(False)
    data_error = (flagged & (evidence["log_move"].abs() > 0.15)
                  & (evidence["reversal"].abs() < 0.1 * evidence["log_move"].abs())
                  & (evidence["vol_ratio"] < 0.5) & (evidence["market_share"] < 0.5))
    reversal_of_error = flagged & data_error.groupby(df["ticker"]).shift(1, fill_value=False)
    corporate = (flagged & (evidence["factor_step"].abs() > 0.01) & evidence["factor_holds"]
                 & ~evidence["adj_return_outlier"])
    market = flagged & (evidence["market_share"] >= 0.5)

    verdict = pd.Series(
        np.select(
            [data_error, reversal_of_error, corporate, market, flagged],
            ["data_error", "data_error_reversal", "corporate_action", "market_event", "genuine_unconfirmed"],
            default="",
        ),
        index=df.index,
    )
    for (ticker, date), v in (manual or {}).items():
        verdict[(df["ticker"] == ticker) & (df["ts"] == pd.Timestamp(date))] = v
    return verdict

#### Investigate

The screen is rerun on `clean` (raw closes are unchanged by the earlier steps, so the flags match 1.2), and the evidence
is computed for every row.

In [66]:
clean = clean.drop(columns=["return", "return_mean", "return_std", "return_zscore", "return_outlier",
                            "return_screen"], errors="ignore")
clean["return"] = calculate_daily_returns(clean)
clean = clean.join(flag_return_outliers(clean))
evidence = return_evidence(clean)

ev_cols = ["ticker", "ts", "close", "return", "return_zscore", "reversal", "vol_ratio", "factor_step",
           "adj_return_outlier", "market_share"]
investigate = clean.join(evidence)
extreme = investigate[investigate["return_zscore"].abs() >= 10]
print(f"Flagged returns: {int(clean['return_outlier'].sum()):,}; with |z| >= 10: {len(extreme)}")
extreme[ev_cols].sort_values("return_zscore", key=abs, ascending=False).head(20).round(3)

Flagged returns: 5,056; with |z| >= 10: 63


,ticker,ts,close,return,return_zscore,reversal,vol_ratio,factor_step,adj_return_outlier,market_share
319538,TRV,1981-08-10,12.060,1.000,41.245,0.000,0.000,0.338,True,0.056
318332,TRV,1976-10-28,9.220,1.000,41.245,0.017,0.000,0.637,True,0.059
318873,TRV,1978-12-20,8.030,0.998,41.142,0.017,0.000,0.546,True,0.118
213788,MCD,1968-01-02,0.175,-0.754,-40.720,-1.396,12.086,-0.009,True,0.077
296654,PG,2000-03-07,29.780,-0.314,-23.196,-0.411,25.779,0.000,True,0.25
319539,TRV,1981-08-11,6.030,-0.500,-20.658,-0.677,0.124,-0.252,True,0.053
293525,PG,1987-10-19,3.810,-0.277,-20.491,-0.125,3.318,0.006,True,1.0
318333,TRV,1976-10-29,4.690,-0.491,-20.300,-0.682,0.500,-0.389,True,0.059
318874,TRV,1978-12-21,4.090,-0.491,-20.272,-0.675,0.727,-0.352,True,0.059
4999,AAPL,2000-09-29,0.460,-0.519,-18.666,-0.791,9.462,0.000,True,0.036


**Case study: MCD on 1968-01-02 (−75%).** It does not reverse, the adjustment factor barely moves, and other
stocks were calm that day. So by the rules above it would count as "genuine". But the surrounding history tells a different
story. The price drops by about 4x and stays there, while normal daily volume rises by about the same factor:

In [67]:
mcd = clean[clean["ticker"] == "MCD"].set_index("ts")
before, after = mcd.loc[:"1967-12-29"].tail(60), mcd.loc["1968-01-02":].head(60)
mcd_break = pd.DataFrame({
    "median close": [before["close"].median(), after["close"].median()],
    "median volume": [before["volume"].median(), after["volume"].median()],
}, index=["60 days before", "60 days after"])
mcd_break.loc["after / before"] = mcd_break.loc["60 days after"] / mcd_break.loc["60 days before"]
print(f"Close 1967-12-29 -> 1968-01-02: {mcd.loc['1967-12-29', 'close']} -> {mcd.loc['1968-01-02', 'close']} "
      f"(÷{mcd.loc['1967-12-29', 'close'] / mcd.loc['1968-01-02', 'close']:.2f})")
print(f"adj_factor: {mcd.loc['1967-12-29', 'adj_factor']:.4f} -> {mcd.loc['1968-01-02', 'adj_factor']:.4f}")
mcd_break.round(4)

Close 1967-12-29 -> 1968-01-02: 0.70988 -> 0.1749 (÷4.06)
adj_factor: 0.4381 -> 0.4341


,median close,median volume
60 days before,0.6111,8.626500e+05
60 days after,0.1628,3.499023e+06
after / before,0.2664,4.056100e+00


Price falls ~4x and the typical share count traded rises ~4x, permanently, while the adjustment factor stays the same.
That is the signature of a roughly **4-for-1 share split that was never applied to the earlier history**. A real 75%
crash would not make the stock trade 4x as many shares from then on. The supplied reference table has the same numbers,
so it cannot correct this. The exact split ratio cannot be verified from this data (÷4.06 on the day is not exactly 4).

**Decision:** verdict `series_break`. Interpolation cannot fix it, because it is a permanent level shift, not a one-day spike.
Rescaling the earlier history would mean guessing an unverified factor. So MCD's history **before** 1968-01-02 is
**quarantined** (the earlier prices are on a different scale from everything after). The history from 1968-01-02 on
is consistent and kept.

In [68]:
MANUAL_VERDICTS = {
    ("MCD", "1968-01-02"): "series_break",  # evidence above: price ÷~4 and volume ×~4, factor unchanged
}
clean["return_verdict"] = classify_extreme_returns(clean, evidence, MANUAL_VERDICTS)

verdicts = clean.loc[clean["return_outlier"].fillna(False)]
summary = verdicts.groupby("return_verdict").agg(
    days=("ts", "size"), tickers=("ticker", "nunique"),
    median_abs_z=("return_zscore", lambda z: z.abs().median()),
).reindex(["data_error", "data_error_reversal", "corporate_action", "series_break", "market_event",
           "genuine_unconfirmed"]).fillna(0).astype({"days": int, "tickers": int})
summary.round(2)

,days,tickers,median_abs_z
return_verdict,,,
data_error,3,1,41.24
data_error_reversal,3,1,20.30
corporate_action,5,5,3.62
series_break,1,1,40.72
market_event,572,30,4.66
genuine_unconfirmed,4472,30,3.63


Each automatic verdict, with its evidence. First the data errors and their reversals:

In [69]:
investigate = clean.join(evidence)
investigate.loc[investigate["return_verdict"].isin(["data_error", "data_error_reversal"]),
                ["ticker", "ts", "open", "high", "low", "close", "volume", "return", "reversal", "vol_ratio",
                 "adj_factor", "return_verdict"]].round(4)

,ticker,ts,open,high,low,close,volume,return,reversal,vol_ratio,adj_factor,return_verdict
318332,TRV,1976-10-28,9.22,9.22,9.22,9.22,0,1.0000,0.0172,0.0000,0.3482,data_error
318333,TRV,1976-10-29,4.69,4.69,4.69,4.69,79200,-0.4913,-0.6824,0.5000,0.2126,data_error_reversal
318873,TRV,1978-12-20,8.03,8.03,8.03,8.03,0,0.9975,0.0173,0.0000,0.3487,data_error
318874,TRV,1978-12-21,4.09,4.09,4.09,4.09,105600,-0.4907,-0.6746,0.7273,0.2259,data_error_reversal
319538,TRV,1981-08-10,12.06,12.06,12.06,12.06,0,1.0000,0.0000,0.0000,0.3483,data_error
319539,TRV,1981-08-11,6.03,6.03,6.03,6.03,24800,-0.5000,-0.6767,0.1238,0.2604,data_error_reversal


All three are TRV, and they share every warning sign. The close is exactly **double** the previous day. The bar is flat
(O = H = L = C) on **zero volume**. The next day the price returns to its earlier level (reversal ≈ 0). The vendor's
adjustment factor also jumps for that single day (0.348 against ~0.213 on both sides). A real doubling would not come
with no trading and undo itself overnight. **Verdict: data error.**

Why the rule also requires low volume *and* a calm market: real crash days can partly reverse the next day too.
UNH on 2008-09-29 (−19%, during the market-wide selloff with 70% of tickers flagged) bounced back the next day
on roughly normal volume. A looser rule would have called it an error and overwritten a real price.

Corporate actions: the raw price jumped, but the adjustment factor changed at the same time, so the adjusted return is normal:

In [70]:
investigate.loc[investigate["return_verdict"] == "corporate_action",
                ["ticker", "ts", "close", "return", "factor_step", "adj_return_outlier"]].round(4)

,ticker,ts,close,return,factor_step,adj_return_outlier
105958,DIS,1974-01-20,0.9150,0.0959,-0.0642,False
192020,JPM,2002-10-02,18.2500,-0.0755,0.0173,False
264448,MSFT,2004-11-15,27.3900,-0.0861,0.1147,False
345722,VZ,1987-12-24,15.1900,-0.0471,0.0159,False
355615,WMT,1974-09-12,0.0098,-0.0547,0.0352,False


Market events, the dates with the most flagged tickers:

In [71]:
me = investigate[investigate["return_verdict"] == "market_event"]
me.groupby("ts").agg(tickers_flagged=("ticker", "size"), share_flagged=("market_share", "first"),
                     median_return=("return", "median")).sort_values("tickers_flagged", ascending=False).head(10).round(3)

,tickers_flagged,share_flagged,median_return
ts,,,
2020-03-12,29,0.967,-0.095
2008-10-28,28,0.933,0.105
2020-03-16,28,0.933,-0.128
2008-10-13,26,0.867,0.129
2020-03-24,25,0.833,0.128
1987-10-19,24,1.0,-0.210
2008-10-15,22,0.733,-0.088
2008-12-01,22,0.733,-0.081
2020-03-13,21,0.7,0.099


The largest `genuine_unconfirmed` moves. Nearly all trade at many times normal volume and do not reverse, which is
how real news looks and the opposite of the TRV errors. They are kept and flagged, without claiming what the news was:

In [72]:
gu = investigate[investigate["return_verdict"] == "genuine_unconfirmed"]
print(f"genuine_unconfirmed: {len(gu):,} days; median volume ratio {gu['vol_ratio'].median():.2f}x; "
      f"share with volume above normal: {(gu['vol_ratio'] > 1).mean():.1%}")
gu.reindex(gu["return_zscore"].abs().sort_values(ascending=False).index)[
    ["ticker", "ts", "return", "return_zscore", "reversal", "vol_ratio", "market_share"]].head(10).round(3)

genuine_unconfirmed: 4,472 days; median volume ratio 1.94x; share with volume above normal: 90.2%


,ticker,ts,return,return_zscore,reversal,vol_ratio,market_share
296654,PG,2000-03-07,-0.314,-23.196,-0.411,25.779,0.25
4999,AAPL,2000-09-29,-0.519,-18.666,-0.791,9.462,0.036
254653,MRK,2004-09-30,-0.268,-17.099,-0.303,28.105,0.034
145854,HON,2000-10-20,0.282,15.454,0.331,8.938,0.107
243833,MMM,2024-07-26,0.230,15.430,0.191,10.377,0.033
101756,CVX,2020-03-18,-0.221,-13.693,-0.208,1.544,0.4
130040,HD,2000-10-12,-0.287,-13.527,-0.299,13.882,0.179
125957,HD,1984-08-17,-0.275,-12.964,-0.362,40.801,0.045
333395,UNH,1996-07-11,-0.298,-12.013,-0.289,12.345,0.04
193606,JPM,2009-01-21,0.251,11.943,0.244,3.145,0.133


#### Repair confirmed data errors: interpolate from neighboring days

For each `data_error` day, every raw price is linearly interpolated **in time** between the previous and next trading day
of the same ticker. A weighted average of two valid bars is itself a valid bar (`low <= open, close <= high` still holds),
so no extra adjustment is needed.

Safeguards:
- **Both neighbors must exist and must not be errors themselves.** No extrapolation at a ticker's first or last day.
  Otherwise the row is quarantined.
- **Adjusted close:** the vendor factor on the error day is itself wrong (it jumps for one day), so it is not used.
  If the factors on the two neighboring days agree (within 0.1%), there was no corporate action between them, and the
  interpolated factor is used. Otherwise the row is quarantined: interpolating across an unverified corporate action
  would be guessing.
- **Volume** is left as reported (0). The day is still a no-trade day.
- Original values are kept (`high_vendor`, `low_vendor`, plus the existing `open_vendor`, `close_vendor`), and the
  repair is flagged in `return_error_repaired`.

In [73]:
def interpolate_error_days(df: pd.DataFrame, is_error: pd.Series) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Linearly interpolate (in time) OHLC and the adjustment factor on single-day data errors.

    ``df`` must be sorted by ticker and date. Error days without two valid neighbors, or whose
    neighbors' adjustment factors disagree by more than 0.1%, are left unchanged and returned in
    quarantine format.
    """
    out = df.copy()
    g = out.groupby("ticker")
    prev_ok = g["ts"].shift(1).notna() & ~is_error.groupby(out["ticker"]).shift(1, fill_value=True)
    next_ok = g["ts"].shift(-1).notna() & ~is_error.groupby(out["ticker"]).shift(-1, fill_value=True)
    f_prev, f_next = g["adj_factor"].shift(1), g["adj_factor"].shift(-1)
    factor_ok = (f_next / f_prev - 1).abs() < 1e-3

    fixable = is_error & prev_ok & next_ok & factor_ok
    t_prev, t_next = g["ts"].shift(1), g["ts"].shift(-1)
    w = (out["ts"] - t_prev) / (t_next - t_prev)  # position of today between the neighbors in time

    for col in ["high", "low"]:
        out[f"{col}_vendor"] = out[col]
    for col in RAW_PRICE_COLS + ["adj_factor"]:
        interp = g[col].shift(1) + w * (g[col].shift(-1) - g[col].shift(1))
        out.loc[fixable, col] = interp[fixable]
    out.loc[fixable, "adj_close"] = out["close"] * out["adj_factor"]
    out["return_error_repaired"] = fixable

    unresolved = is_error & ~fixable
    reason = np.where(~(prev_ok & next_ok)[unresolved], "data error without two valid neighbors",
                      "data error, neighbors' adjustment factors disagree")
    return out, quarantine_rows(out, unresolved, "").assign(reason=reason)

Check on an injected series: a doubled flat bar on zero volume between two normal days (repairable), the same
error on the last day (no right neighbor → quarantined), and one where the factor changes between the neighbors
(possible corporate action → quarantined). A gap of 3 calendar days on one side tests the time weighting.

In [74]:
demo = pd.DataFrame({
    "ticker": ["AAA"] * 3 + ["BBB"] * 2 + ["CCC"] * 3,
    "ts": pd.to_datetime(["2024-01-04", "2024-01-05", "2024-01-08",   # Fri -> Mon gap after the error
                          "2024-01-04", "2024-01-05",
                          "2024-01-04", "2024-01-05", "2024-01-08"]),
    "open":  [10.0, 20.0, 10.4,  5.0, 10.0,  8.0, 16.0, 8.0],
    "high":  [10.2, 20.0, 10.6,  5.1, 10.0,  8.1, 16.0, 8.1],
    "low":   [ 9.8, 20.0, 10.2,  4.9, 10.0,  7.9, 16.0, 7.9],
    "close": [10.0, 20.0, 10.4,  5.0, 10.0,  8.0, 16.0, 8.0],
    "adj_factor": [0.5, 0.9, 0.5,  1.0, 1.0,  0.5, 0.9, 0.6],
})
demo["adj_close"] = demo["close"] * demo["adj_factor"]
demo_err = demo["ts"].eq("2024-01-05")
demo_out, demo_q = interpolate_error_days(demo, demo_err)

aaa = demo_out[demo_out["ticker"] == "AAA"].iloc[1]
assert np.isclose(aaa["close"], 10.0 + 0.25 * 0.4)        # 1 of 4 calendar days from 01-04 to 01-08
assert np.isclose(aaa["adj_factor"], 0.5) and np.isclose(aaa["adj_close"], aaa["close"] * 0.5)
assert aaa["low"] <= min(aaa["open"], aaa["close"]) and aaa["high"] >= max(aaa["open"], aaa["close"])
assert demo_q["ticker"].tolist() == ["BBB", "CCC"]
display(demo_out[["ticker", "ts", "open", "high", "low", "close", "adj_factor", "adj_close", "return_error_repaired"]])
demo_q

,ticker,ts,open,high,low,close,adj_factor,adj_close,return_error_repaired
0,AAA,2024-01-04,10.0,10.2,9.8,10.0,0.5,5.00,False
1,AAA,2024-01-05,10.1,10.3,9.9,10.1,0.5,5.05,True
2,AAA,2024-01-08,10.4,10.6,10.2,10.4,0.5,5.20,False
3,BBB,2024-01-04,5.0,5.1,4.9,5.0,1.0,5.00,False
4,BBB,2024-01-05,10.0,10.0,10.0,10.0,1.0,10.00,False
5,CCC,2024-01-04,8.0,8.1,7.9,8.0,0.5,4.00,False
6,CCC,2024-01-05,16.0,16.0,16.0,16.0,0.9,14.40,False
7,CCC,2024-01-08,8.0,8.1,7.9,8.0,0.6,4.80,False


,ticker,ts,reason
4,BBB,2024-01-05,data error without two valid neighbors
6,CCC,2024-01-05,"data error, neighbors' adjustment factors disa..."


#### Apply to `ohlc_hw`

In [75]:
is_error = clean["return_verdict"] == "data_error"
clean, error_q = interpolate_error_days(clean, is_error)

# Series breaks: quarantine the ticker's history before the break (different price scale, unverified factor).
break_q = []
for _, row in clean[clean["return_verdict"] == "series_break"].iterrows():
    earlier = (clean["ticker"] == row["ticker"]) & (clean["ts"] < row["ts"])
    break_q.append(quarantine_rows(clean, earlier, f"before unadjusted series break on {row['ts'].date()} "
                                                   "(price scale ~4x different, split ratio unverified)"))
quarantine = pd.concat([q for q in [quarantine, error_q, *break_q] if len(q)], ignore_index=True)

print(f"Data errors repaired by interpolation: {int(clean['return_error_repaired'].sum())}")
print(f"Data errors quarantined:               {len(error_q)}")
print(f"Rows quarantined for series breaks:    {sum(len(q) for q in break_q)}")
clean.loc[clean["return_error_repaired"],
          ["ticker", "ts", "open_vendor", "close_vendor", "open", "high", "low", "close", "adj_factor", "adj_close"]]

Data errors repaired by interpolation: 2
Data errors quarantined:               1
Rows quarantined for series breaks:    376


,ticker,ts,open_vendor,close_vendor,open,high,low,close,adj_factor,adj_close
318332,TRV,1976-10-28,9.22,9.22,4.65,4.65,4.65,4.65,0.212638,0.988767
319538,TRV,1981-08-10,12.06,12.06,6.03,6.03,6.03,6.03,0.260365,1.570000


#### Recheck after repair

Recompute returns on the repaired data, excluding quarantined rows, and rerun the 3-std screen. Then confirm that the errors and their reversals are
no longer extreme, the bars are valid, and adjusted close is still `close × adj_factor`.

In [76]:
in_q = clean.set_index(["ticker", "ts"]).index.isin(quarantine.set_index(["ticker", "ts"]).index)
kept = clean[~in_q].copy()
kept["return"] = calculate_daily_returns(kept)
recheck = kept.join(flag_return_outliers(kept)[["return_zscore", "return_outlier"]], rsuffix="_after")

was_error = kept["return_verdict"].isin(["data_error", "data_error_reversal"])
print("Former errors / reversals after repair:")
print(recheck.loc[was_error, ["ticker", "ts", "close", "return", "return_zscore_after", "return_outlier_after"]]
      .round(4).to_string())

assert not recheck.loc[was_error, "return_outlier_after"].any(), "a repaired error is still an outlier"
assert not flag_ohlc_bounds(kept)["ohlc_inconsistent"].any(), "OHLC bounds broken"
assert np.allclose(kept["adj_close"], kept["close"] * kept["adj_factor"], rtol=1e-6), "adj_close inconsistent"
assert kept.loc[kept["ticker"] == "MCD", "ts"].min() == pd.Timestamp("1968-01-02")
print(f"\nOK. Flagged returns after repair: {int(recheck['return_outlier_after'].sum()):,} "
      f"(before: {int(clean['return_outlier'].sum()):,}); MCD now starts 1968-01-02.")

flag_change = pd.DataFrame({
    "flagged_before": clean.groupby("ticker")["return_outlier"].sum(),
    "flagged_after": recheck.groupby("ticker")["return_outlier_after"].sum(),
    "std_before": clean.groupby("ticker")["return_std"].first(),
    "std_after": flag_return_outliers(kept).groupby(kept["ticker"])["return_std"].first(),
}).astype({"flagged_before": int, "flagged_after": int})
flag_change["change"] = flag_change["flagged_after"] - flag_change["flagged_before"]
flag_change[flag_change["change"] != 0].round(4)

Former errors / reversals after repair:
       ticker         ts  close  return  return_zscore_after  return_outlier_after
318332    TRV 1976-10-28   4.65  0.0087               0.4696                 False
318333    TRV 1976-10-29   4.69  0.0086               0.4654                 False
318874    TRV 1978-12-21   4.09  0.0174               0.9691                 False
319538    TRV 1981-08-10   6.03  0.0000              -0.0264                 False
319539    TRV 1981-08-11   6.03  0.0000              -0.0264                 False

OK. Flagged returns after repair: 5,200 (before: 5,056); MCD now starts 1968-01-02.


,flagged_before,flagged_after,std_before,std_after,change
ticker,,,,,
MCD,180,205,0.0185,0.0171,25
TRV,87,206,0.0242,0.0175,119


**Result.**
- **Data errors (3, all TRV):** one-day doubled prices on zero-volume flat bars, each reversed the next day.
  - **2 repaired** (1976-10-28, 1981-08-10) by time-weighted interpolation of OHLC and the adjustment factor from the
    neighboring days. Their reversal days are fixed with them, and none is extreme any more.
  - **1 quarantined** (1978-12-20). The adjustment factor differs by 0.2% between its neighbors (0.22549 → 0.22594).
    That is small, but it is above the 0.1% tolerance, and a distribution between them cannot be ruled out. Quarantining
    is the documented choice rather than interpolating across a possible corporate action. The next day's return is
    then measured from 12-19 and is normal.
- **Series break (MCD, 1968-01-02):** an unadjusted ~4-for-1 split. The 376 earlier MCD rows are quarantined, because
  their scale cannot be reconciled with an unverified ratio.
- **Corporate actions, market events, and genuine (unconfirmed) moves are kept.** They stay flagged in the cleaned data
  through `return_outlier`, `return_zscore`, and `return_verdict`.
- **Why the flag count rises after repair** (table above): the screen is relative to each ticker's own std. Removing
  TRV's ±100% errors and MCD's −75% break shrinks those two tickers' std sharply, so more of their ordinary large days
  now exceed 3 std. Other tickers are unchanged. This is the 3-std rule working as defined, not new errors. Those
  extra days are review flags only.

**Uncertainties stated, not hidden:** mergers and specific news cannot be identified without an external events
source; `genuine_unconfirmed` means only "no evidence of error". The rule thresholds (15% move, 90% reversal, volume
under 0.5x, not market-wide, 1% factor step) are judgment calls. They were chosen so that the clear errors are caught
while real moves that partly reverse, such as UNH on 2008-09-29, are not.

### Validation and saving `ohlc_hw_cleaned`

Steps:
1. **Split** `clean` into retained rows and quarantined rows (one quarantine reason per row; a row is listed once even
   if several steps flagged it).
2. **Account for every raw row:** raw = exact duplicates removed + quarantined + retained.
3. **Choose the saved columns:** the eight data columns plus every repair/audit flag. `ts` is stored as an ISO string
   `YYYY-MM-DD` so SQL date filters such as `ts BETWEEN '2024-01-01' AND '2024-12-31'` compare correctly.
4. **Write** `ohlc_hw_cleaned`, plus `ohlc_hw_quarantine` with the reasons, to the local working database. The raw
   `ohlc_hw` table is not touched.
5. **Validate the table as read back from the database**, not the in-memory frame, so the checks cover what was actually saved.
6. **Before/after statistics.**

#### 1–2. Retained vs. quarantined, and row accounting

In [77]:
quarantine_final = (quarantine.drop_duplicates(["ticker", "ts"])  # first reason recorded wins
                    .sort_values(["ticker", "ts"]).reset_index(drop=True))
q_keys = pd.MultiIndex.from_frame(quarantine_final[["ticker", "ts"]])
is_quarantined = pd.MultiIndex.from_frame(clean[["ticker", "ts"]]).isin(q_keys)
retained = clean[~is_quarantined].copy()

n_dupes = len(raw) - len(prices)
accounting = pd.Series({
    "raw rows in ohlc_hw": len(raw),
    "exact duplicates removed": n_dupes,
    "unique (ticker, ts) rows": len(prices),
    "rows dropped (all raw prices missing)": len(dropped),
    "quarantined": int(is_quarantined.sum()),
    "retained in ohlc_hw_cleaned": len(retained),
}, name="rows")
assert accounting["raw rows in ohlc_hw"] == n_dupes + len(clean) + len(dropped)
assert len(clean) == accounting["quarantined"] + accounting["retained in ohlc_hw_cleaned"]
print(accounting.map("{:,}".format).to_string())
print()
print("Quarantine reasons:")
print(quarantine_final["reason"].value_counts().to_string())

raw rows in ohlc_hw                      397,711
exact duplicates removed                  29,460
unique (ticker, ts) rows                 368,251
rows dropped (all raw prices missing)          0
quarantined                                  377
retained in ohlc_hw_cleaned              367,874

Quarantine reasons:
reason
before unadjusted series break on 1968-01-02 (price scale ~4x different, split ratio unverified)    376
data error, neighbors' adjustment factors disagree                                                    1


#### 3. Columns saved

The return screen is rerun on the retained rows, so `return_outlier` and `return_zscore` describe the **saved** data.
The verdicts from the investigation stay in `return_verdict`, and the original 1.2 flag is kept as `return_outlier_before`.
Flags that the rerun adds (from the smaller TRV/MCD std, explained above) have no investigated verdict and are labelled
`flagged_after_repair`.

In [78]:
retained = retained.rename(columns={"return_outlier": "return_outlier_before"})
retained["return"] = calculate_daily_returns(retained)
after = flag_return_outliers(retained)
retained["return_outlier"] = after["return_outlier"].fillna(False).astype(bool)
retained["return_zscore"] = after["return_zscore"]
retained["return_outlier_before"] = retained["return_outlier_before"].fillna(False).astype(bool)
new_flag = retained["return_outlier"] & (retained["return_verdict"] == "")
retained.loc[new_flag, "return_verdict"] = "flagged_after_repair"
retained["zero_volume"] = stale_zero_volume(retained)

CLEANED_COLS = (
    ["ts", "ticker"] + PRICE_COLS + ["volume"]
    # audit of ticker normalization and the original vendor values
    + ["ticker_raw", "open_vendor", "high_vendor", "low_vendor", "close_vendor", "adj_close_vendor"]
    # repair flags and methods
    + [f"{c}_ffilled" for c in RAW_PRICE_COLS] + ["volume_filled", "adj_factor", "adj_close_source",
       "open_repaired", "close_repaired", "price_repair_method", "return_error_repaired"]
    # review flags (kept data, flagged)
    + ["return_outlier", "return_zscore", "return_outlier_before", "return_verdict", "zero_volume"]
)
FLAG_COLS = [c for c in CLEANED_COLS if retained[c].dtype == bool]

to_save = retained[CLEANED_COLS].copy()
to_save["ts"] = to_save["ts"].dt.strftime("%Y-%m-%d")
to_save[FLAG_COLS] = to_save[FLAG_COLS].astype(int)  # SQLite has no boolean type: store 0/1

q_save = quarantine_final.merge(clean, on=["ticker", "ts"], how="left")[
    ["ts", "ticker", "reason"] + PRICE_COLS + ["volume", "ticker_raw"]]
q_save["ts"] = q_save["ts"].dt.strftime("%Y-%m-%d")

print(f"{len(CLEANED_COLS)} columns; flag columns stored as 0/1: {len(FLAG_COLS)}")
to_save.head()

30 columns; flag columns stored as 0/1: 11


,ts,ticker,open,high,low,close,adj_close,volume,ticker_raw,open_vendor,...,adj_close_source,open_repaired,close_repaired,price_repair_method,return_error_repaired,return_outlier,return_zscore,return_outlier_before,return_verdict,zero_volume
0,1980-12-12,AAPL,0.12835,0.12835,0.12835,0.12835,0.098941,469034068,AAPL,0.12835,...,vendor,0,0,,0,0,NaN,0,,0
1,1980-12-15,AAPL,0.12165,0.12165,0.12165,0.12165,0.093779,175884971,AAPL,0.12165,...,vendor,0,0,,0,0,-1.913208,0,,0
2,1980-12-16,AAPL,0.11272,0.11272,0.11272,0.11272,0.086896,105728103,AAPL,0.11272,...,vendor,0,0,,0,0,-2.674763,0,,0
3,1980-12-17,AAPL,0.11551,0.11551,0.11551,0.11551,0.089046,86441684,AAPL,0.11551,...,vendor,0,0,,0,0,0.850292,0,,0
4,1980-12-18,AAPL,0.11886,0.11886,0.11886,0.11886,0.091627,73449671,AAPL,0.11886,...,vendor,0,0,,0,0,1.002925,0,,0


#### 4. Write to the database

In [79]:
with sqlite3.connect(DB_PATH) as conn:
    raw_before = pd.read_sql_query("SELECT * FROM ohlc_hw", conn)
    to_save.to_sql("ohlc_hw_cleaned", conn, if_exists="replace", index=False)
    q_save.to_sql("ohlc_hw_quarantine", conn, if_exists="replace", index=False)
    conn.execute("CREATE UNIQUE INDEX IF NOT EXISTS ix_ohlc_hw_cleaned_key ON ohlc_hw_cleaned (ticker, ts)")
    raw_after = pd.read_sql_query("SELECT * FROM ohlc_hw", conn)
    cleaned_db = pd.read_sql_query("SELECT * FROM ohlc_hw_cleaned", conn)
    tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name", conn)

assert raw_after.equals(raw_before) and raw_after.equals(raw), "raw ohlc_hw table changed"
print("Tables:", tables["name"].tolist())
print(f"ohlc_hw_cleaned: {len(cleaned_db):,} rows; ohlc_hw_quarantine: {len(q_save):,} rows; raw ohlc_hw unchanged")

Tables: ['market_caps', 'ohlc', 'ohlc_hw', 'ohlc_hw_cleaned', 'ohlc_hw_quarantine']
ohlc_hw_cleaned: 367,874 rows; ohlc_hw_quarantine: 377 rows; raw ohlc_hw unchanged


#### 5. Validate the saved table

Every check runs on `cleaned_db`, the table read back from SQLite. The unique index created above also makes the
database itself reject any duplicate `(ticker, ts)`.

In [80]:
v = cleaned_db.assign(ts=pd.to_datetime(cleaned_db["ts"], format="%Y-%m-%d"))
raw_prices = v[RAW_PRICE_COLS]
checks = {
    "no missing raw or adjusted prices": v[PRICE_COLS].notna().all().all(),
    "all prices finite": np.isfinite(v[PRICE_COLS]).all().all(),
    "all prices positive": (v[PRICE_COLS] > 0).all().all(),
    "unique (ticker, ts) keys": not v.duplicated(["ticker", "ts"]).any(),
    "volume present, finite, nonnegative": (v["volume"].notna() & np.isfinite(v["volume"]) & (v["volume"] >= 0)).all(),
    "low <= open <= high": ((v["low"] <= v["open"]) & (v["open"] <= v["high"])).all(),
    "low <= close <= high": ((v["low"] <= v["close"]) & (v["close"] <= v["high"])).all(),
    "low <= high": (v["low"] <= v["high"]).all(),
    "adj_close = close x adj_factor": np.allclose(v["adj_close"], v["close"] * v["adj_factor"], rtol=1e-6, atol=0),
    "ts stored as ISO YYYY-MM-DD": cleaned_db["ts"].str.fullmatch(r"\d{4}-\d{2}-\d{2}").all(),
    "tickers normalized (no markup/whitespace/lowercase)": v["ticker"].str.fullmatch(r"[A-Z]+").all(),
    "rows sorted by ticker and date": v.sort_values(["ticker", "ts"]).equals(v),
}
validation = pd.Series(checks, name="passed")
print(validation.to_string())
assert validation.all(), validation[~validation]

no missing raw or adjusted prices                      True
all prices finite                                      True
all prices positive                                    True
unique (ticker, ts) keys                               True
volume present, finite, nonnegative                    True
low <= open <= high                                    True
low <= close <= high                                   True
low <= high                                            True
adj_close = close x adj_factor                         True
ts stored as ISO YYYY-MM-DD                            True
tickers normalized (no markup/whitespace/lowercase)    True
rows sorted by ticker and date                         True


SQL date filters work on the saved `ts` column, as Part 2 will need. Also, as a final check, the missing-trading-day
detector is rerun: any gap now must be one of the quarantined rows, never an unexplained one.

In [81]:
with sqlite3.connect(DB_PATH) as conn:
    sql_check = pd.read_sql_query(
        "SELECT ticker, COUNT(*) AS days, MIN(ts) AS first, MAX(ts) AS last FROM ohlc_hw_cleaned "
        "WHERE ticker = 'AAPL' AND ts BETWEEN '2024-01-01' AND '2024-12-31' GROUP BY ticker", conn)
print(sql_check.to_string(index=False))

# Gaps relative to the reference calendar, over each ticker's retained history.
gaps_after = find_missing_trading_days(v[["ticker", "ts"]], ref_calendar)
gaps_explained = gaps_after.merge(quarantine_final, on=["ticker", "ts"], how="left")
assert gaps_explained["reason"].notna().all(), "unexplained missing trading day"
print(f"\nMissing trading days in cleaned data: {len(gaps_after)} (all quarantined rows):")
gaps_explained

ticker  days      first       last
  AAPL   202 2024-01-02 2024-10-18

Missing trading days in cleaned data: 1 (all quarantined rows):


,ticker,ts,reason
0,TRV,1978-12-20,"data error, neighbors' adjustment factors disa..."


#### 6. Before/after statistics

In [82]:
def issue_counts(df: pd.DataFrame, raw_labels: pd.Series | None = None) -> pd.Series:
    """Headline data-quality counts for a price table (``ts`` as datetime, tickers as given)."""
    norm = df.assign(ticker=df["ticker"].map(normalize_ticker))
    adj_ok = np.isclose(df["adj_close"], df["close"] * df["adj_factor"], rtol=1e-6) if "adj_factor" in df else None
    return pd.Series({
        "rows": len(df),
        "distinct ticker labels": df["ticker"].nunique(),
        "exact duplicate rows": int(norm.duplicated(subset=DATA_COLS).sum()),
        "missing adj_close": int(df["adj_close"].isna().sum()),
        "nonpositive prices": int((df[PRICE_COLS] <= 0).sum().sum()),
        "OHLC bound violations": int(flag_ohlc_bounds(df)["ohlc_inconsistent"].sum()),
        "zero-volume days": int((df["volume"] == 0).sum()),
    })


raw_for_stats = raw.assign(ts=pd.to_datetime(raw["ts"], format="%Y-%m-%d"))
before_after = pd.DataFrame({"before (ohlc_hw)": issue_counts(raw_for_stats),
                             "after (ohlc_hw_cleaned)": issue_counts(v)})

# adj_close inconsistent with the reference factor (before: vendor values matched to ref; after: saved values).
vendor_check = prices[["ticker", "ts", "close", "adj_close"]].merge(ref_factors[["ticker", "ts", "adj_factor"]])
before_after.loc["adj_close inconsistent with close x factor"] = [
    int((vendor_check["adj_close"].notna()
         & ~np.isclose(vendor_check["adj_close"], vendor_check["close"] * vendor_check["adj_factor"], rtol=1e-6)).sum()),
    int((~np.isclose(v["adj_close"], v["close"] * v["adj_factor"], rtol=1e-6)).sum()),
]
before_after.loc["one-day data-error spikes (TRV)"] = [3, 0]
before_after

,before (ohlc_hw),after (ohlc_hw_cleaned)
rows,397711,367874
distinct ticker labels,150,30
exact duplicate rows,29460,0
missing adj_close,19855,0
nonpositive prices,5,0
OHLC bound violations,5,0
zero-volume days,81,74
adj_close inconsistent with close x factor,10417,0
one-day data-error spikes (TRV),3,0


Summary statistics of the price and volume columns. Before = raw table as loaded (including its 29,460 duplicate rows, which is also why its zero-volume count above is 81 rather than the 75 unique days); after = saved cleaned table:

In [83]:
stats = pd.concat(
    {"before": raw[PRICE_COLS + ["volume"]].describe(), "after": v[PRICE_COLS + ["volume"]].describe()},
    axis=1,
).swaplevel(axis=1).sort_index(axis=1, level=0, sort_remaining=False)
stats[[(c, s) for c in PRICE_COLS + ["volume"] for s in ["before", "after"]]].T.round(4)

count          mean           std     min           25%         50%          75%           max
open      before  397711.0  4.628550e+01  6.713260e+01  0.0000  4.209900e+00       19.85        57.53  6.042600e+02
          after   367874.0  4.635580e+01  6.716130e+01  0.0052  4.230000e+00       19.95        57.62  6.042600e+02
high      before  397711.0  4.674770e+01  6.777820e+01  0.0052  4.240000e+00       20.06        58.13  6.086300e+02
          after   367874.0  4.681870e+01  6.780750e+01  0.0052  4.270000e+00       20.17        58.22  6.086300e+02
low       before  397711.0  4.581630e+01  6.647720e+01  0.0048  4.160000e+00       19.63        56.95  5.978800e+02
          after   367874.0  4.588630e+01  6.650730e+01  0.0048  4.190000e+00       19.73        57.04  5.978800e+02
close     before  397711.0  4.629290e+01  6.714190e+01  0.0052  4.200000e+00       19.86        57.53  6.054000e+02
          after   367874.0  4.636370e+01  6.717300e+01  0.0052  4.230000e+00       19.96        57.63  6.054000e+02
adj_close before  377856.0  5.165230e+01  1.945840e+02  0.0029  1.390000e+00       11.96        43.77  1.044251e+04
          after   367874.0  3.731220e+01  6.360910e+01  0.0029  1.320000e+00       11.50        41.63  6.054000e+02
volume    before  397711.0  3.097773e+07  1.266237e+08  0.0000  1.730700e+06  4648200.00  11023796.50  9.276606e+09
          after   367874.0  3.098225e+07  1.266757e+08  0.0000  1.738768e+06  4657753.00  11033807.75  9.276606e+09

Daily raw-close returns per ticker, before (de-duplicated raw data) and after cleaning:

In [84]:
ret_before = prices.groupby("ticker")["return"].agg(["std", "min", "max"])
ret_after = retained.groupby("ticker")["return"].agg(["std", "min", "max"])
ret_cmp = pd.concat({"before": ret_before, "after": ret_after}, axis=1)
changed = (ret_cmp["before"] - ret_cmp["after"]).abs().max(axis=1) > 1e-12
print(f"Tickers whose return statistics changed: {changed.sum()} of {len(ret_cmp)} -> {ret_cmp.index[changed].tolist()}")
ret_cmp[changed].round(4)

Tickers whose return statistics changed: 2 of 30 -> ['MCD', 'TRV']


before                   after               
           std     min     max     std    min     max
ticker                                               
MCD     0.0185 -0.7536  0.1813  0.0171 -0.197  0.1813
TRV     0.0242 -0.5000  1.0000  0.0175 -0.208  0.2556

What was repaired, by flag, in the saved table:

In [85]:
repairs = pd.Series({
    "adj_close rebuilt (was missing)": int((v["adj_close_source"] == "rebuilt_missing").sum()),
    "adj_close rebuilt (was inconsistent)": int((v["adj_close_source"] == "rebuilt_inconsistent").sum()),
    "open repaired (impossible price)": int(v["open_repaired"].sum()),
    "close repaired (impossible price)": int(v["close_repaired"].sum()),
    "data-error days interpolated": int(v["return_error_repaired"].sum()),
    "raw prices forward-filled": int(v[[f"{c}_ffilled" for c in RAW_PRICE_COLS]].to_numpy().sum()),
    "volume filled with 0": int(v["volume_filled"].sum()),
}, name="rows")
flags = v.loc[v["return_outlier"] == 1, "return_verdict"].value_counts().rename("flagged returns (kept)")
display(repairs.to_frame())
flags.to_frame()

,rows
adj_close rebuilt (was missing),18358
adj_close rebuilt (was inconsistent),10410
open repaired (impossible price),4
close repaired (impossible price),0
data-error days interpolated,2
raw prices forward-filled,0
volume filled with 0,0


,flagged returns (kept)
return_verdict,
genuine_unconfirmed,4449
market_event,572
flagged_after_repair,174
corporate_action,5


**Validation result: every check passes on the table as saved.**

- `ohlc_hw_cleaned` has **367,874 rows** for 30 tickers. Accounting: 397,711 raw = 29,460 exact duplicates removed
  + 377 quarantined (376 MCD rows before its 1968 series break, 1 TRV data error) + 367,874 retained. The quarantined
  rows and their reasons are saved in `ohlc_hw_quarantine`.
- Saved prices are present, finite, and positive, including adjusted close. `(ticker, ts)` is unique, enforced by an
  index. Volume is finite and nonnegative. Every bar satisfies `low <= open, close <= high`, and every adjusted close
  equals `close × adj_factor`.
- The only gaps in the trading calendar are the quarantined rows.
- Biggest before/after changes: ticker labels go from 150 to 30, duplicates from 29,460 to 0, missing `adj_close` from
  19,855 to 0, inconsistent `adj_close` from 10,417 to 0, and `adj_close`'s maximum from ~10,442 (a corrupted value) to a
  plausible level. Only TRV's and MCD's return statistics change, because of the confirmed errors removed.
- Kept-but-flagged data stays visible: return outliers (with verdicts), zero-volume days, and every repair flag are
  columns in the cleaned table. The raw `ohlc_hw` table is verified unchanged.